# 09. SMART Barangs & Lots Fine-Grained Subcategory Consolidation
### Modular, Subcategory-by-Subcategory Deduplication & Master Data Synchronization
---
This notebook serves as the final refinement stage of the AMS-to-SMART data migration:
1. **Granular Normalization**: Iterates through individual subcategories to resolve messy specifications, embedded location strings, and syntax anomalies.
2. **Modular Architecture**: Each subcategory has its own dedicated, self-contained execution cell.
3. **Master Re-Synchronization**: After updating any subcategory, `sync_consolidated_masters()` automatically aggregates all 111 subcategories, re-indexes sequential IDs, re-links units, and re-exports all staging and final deliverables.
4. **Data Integrity Assured**: Every execution re-validates 100% of global referential and schema assertions.

In [1]:
import os
import sys
import json
import re
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Dynamic Path Resolution
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path('/home/jovyan/data'),                      # Docker container path
        Path('data'),                                  # Repository root
        Path('../data'),                               # Notebooks directory
        Path('/home/ran/Devs/ams-smart-migration/data') # Host absolute path
    ]
    for p in candidates:
        if p.exists() and (p / 'smart').exists():
            return p.resolve()
    return Path('data').resolve()

DATA_DIR = get_data_dir()
STAGING_ASSETS_DIR = DATA_DIR / 'smart' / 'staging' / 'assets'
STAGING_UNITS_DIR = DATA_DIR / 'smart' / 'staging' / 'units'
STAGING_BARANGS_DIR = DATA_DIR / 'smart' / 'staging' / 'barangs'
STAGING_LOTS_DIR = DATA_DIR / 'smart' / 'staging' / 'lots'

FINAL_BARANG_DIR = DATA_DIR / 'smart' / 'final' / 'barang'
FINAL_LOT_DIR = DATA_DIR / 'smart' / 'final' / 'lot'
FINAL_UNIT_DIR = DATA_DIR / 'smart' / 'final' / 'unit'

FINAL_SUBCATEGORY_PATH = DATA_DIR / 'smart' / 'final' / 'subcategory' / 'subcategories.csv'
FINAL_BRAND_PATH = DATA_DIR / 'smart' / 'final' / 'brand' / 'brands.csv'
FINAL_VENDOR_PATH = DATA_DIR / 'smart' / 'final' / 'vendor' / 'vendors.csv'

STAGING_BRAND_MAP_PATH = DATA_DIR / 'smart' / 'staging' / 'brand' / 'brand_ams_mapping.csv'
STAGING_VENDOR_MAP_PATH = DATA_DIR / 'smart' / 'staging' / 'vendor' / 'vendor_ams_mapping.csv'
UNITS_MANIFEST_PATH = STAGING_UNITS_DIR / '_units_manifest.json'

for d in [STAGING_BARANGS_DIR, STAGING_LOTS_DIR, FINAL_BARANG_DIR, FINAL_LOT_DIR, FINAL_UNIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. Load Master Lookups
# ---------------------------------------------------------
df_sub = pd.read_csv(FINAL_SUBCATEGORY_PATH, dtype=str)
sub_map = dict(zip(df_sub['code'], df_sub['id']))

df_brand = pd.read_csv(FINAL_BRAND_PATH, dtype=str)
brand_name_to_id = {r['name'].strip().upper(): str(r['id']).strip() for _, r in df_brand.iterrows() if pd.notna(r['name'])}

df_bmap = pd.read_csv(STAGING_BRAND_MAP_PATH, dtype=str)
ams_brand_to_id = {}
for _, r in df_bmap.iterrows():
    if pd.notna(r.get('ams_brand_name')):
        ams_brand_to_id[r['ams_brand_name'].strip().upper()] = str(r['smart_brand_id']).strip()

brand_aliases = {
    'LIONSTAR': brand_name_to_id.get('LION STAR', '125'),
    'LION STAR': brand_name_to_id.get('LION STAR', '125'),
    'PONGS': brand_name_to_id.get('PONGS HOME', '79'),
    'PONGS HOME': brand_name_to_id.get('PONGS HOME', '79'),
    'SYLYART': brand_name_to_id.get('SYLY ART', '130'),
    'SYLU ART': brand_name_to_id.get('SYLY ART', '130'),
    'SOFA': '123',
    'NO BRAND': '123',
    'CHITOSE': '173',
    'ICA': '174',
    'STELA': '175',
    'STELLA': '175',
    'MORELLY': '176',
}

df_vmap = pd.read_csv(STAGING_VENDOR_MAP_PATH, dtype=str)
vendor_map = {}
for _, r in df_vmap.iterrows():
    v_id = str(r['ams_id']).strip()
    if v_id.endswith('.0'): v_id = v_id[:-2]
    vendor_map[v_id] = str(r['smart_vendor_id']).strip()

def resolve_brand_id(raw_brand):
    if pd.isna(raw_brand) or not str(raw_brand).strip():
        return '123'
    b = str(raw_brand).strip().upper()
    if b in brand_aliases:
        return brand_aliases[b]
    if b in brand_name_to_id:
        return brand_name_to_id[b]
    if b in ams_brand_to_id:
        return ams_brand_to_id[b]
    return '123'

def resolve_vendor_id(raw_vendor):
    if pd.isna(raw_vendor) or str(raw_vendor).strip() in ('', 'NULL', 'null', 'None'):
        return None
    v = str(raw_vendor).strip()
    if v.endswith('.0'): v = v[:-2]
    return vendor_map.get(v, None)

def resolve_organizer_id(raw_org):
    return '2' if str(raw_org).strip().upper() == 'ICT' else '1'

def clean_text(s):
    if pd.isna(s) or s is None:
        return ''
    return re.sub(r'\s+', ' ', str(s).replace('\r', ' ').replace('\n', ' ')).strip()

# ---------------------------------------------------------
# 3. Master Re-Synchronization Engine
# ---------------------------------------------------------
def escape_sql(val):
    if val is None or pd.isna(val):
        return 'NULL'
    s = str(val).replace("'", "''")
    return f"N'{s}'"

def format_num_sql(val):
    if val is None or pd.isna(val) or str(val).strip() in ('', 'None', 'nan', 'NULL'):
        return 'NULL'
    try:
        f_val = float(val)
        return str(int(f_val)) if f_val.is_integer() else str(f_val)
    except (ValueError, TypeError):
        return str(val)

def sync_consolidated_masters():
    with open(UNITS_MANIFEST_PATH, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
        
    all_barangs = []
    all_lots = []
    all_units = []
    
    current_id = 1
    barangs_manifest = {'subcategories': {}, 'total_barangs': 0}
    lots_manifest = {'subcategories': {}, 'total_lots': 0}
    units_manifest = {'subcategories': {}, 'total_units': 0}
    
    for sub_code, meta in sorted(manifest['subcategories'].items()):
        b_file = STAGING_BARANGS_DIR / meta['file']
        l_file = STAGING_LOTS_DIR / meta['file']
        u_file = STAGING_UNITS_DIR / meta['file']
        
        if not b_file.exists() or meta['row_count'] == 0:
            continue
            
        df_b = pd.read_csv(b_file, dtype=str)
        df_l = pd.read_csv(l_file, dtype=str)
        df_u = pd.read_csv(u_file, dtype=str)
        if 'specification' not in df_u.columns:
            df_u['specification'] = None
        
        # Re-index IDs sequentially for this partition
        old_b_to_new_id = {}
        for idx in range(len(df_b)):
            old_b_to_new_id[str(df_b.at[idx, 'id'])] = current_id
            df_b.at[idx, 'id'] = current_id
            df_l.at[idx, 'id'] = current_id
            df_l.at[idx, 'barang_id'] = current_id
            current_id += 1
            
        # Save re-indexed partition files
        df_b.to_csv(b_file, index=False)
        df_l.to_csv(l_file, index=False)
        
        # Update units lot_id mapping
        if 'lot_id' in df_u.columns:
            df_u['lot_id'] = df_u['lot_id'].map(lambda x: old_b_to_new_id.get(str(x), x))
            for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
                if c in df_u.columns:
                    df_u[c] = pd.to_numeric(df_u[c], errors='coerce').astype('Int64')
            df_u.to_csv(u_file, index=False)
            
        all_barangs.append(df_b)
        all_lots.append(df_l)
        all_units.append(df_u)
        
        barangs_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_b)}
        lots_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_l)}
        units_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_u)}
        
    df_barangs_master = pd.concat(all_barangs, ignore_index=True)
    df_lots_master = pd.concat(all_lots, ignore_index=True)
    df_units_master = pd.concat(all_units, ignore_index=True)
    
    barangs_manifest['total_barangs'] = len(df_barangs_master)
    lots_manifest['total_lots'] = len(df_lots_master)
    units_manifest['total_units'] = len(df_units_master)
    
    # 1. Staging Masters
    df_barangs_master.to_csv(STAGING_BARANGS_DIR / 'barangs.csv', index=False)
    with open(STAGING_BARANGS_DIR / '_barangs_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(barangs_manifest, f, indent=2)
        
    df_lots_master.to_csv(STAGING_LOTS_DIR / 'lots.csv', index=False)
    with open(STAGING_LOTS_DIR / '_lots_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(lots_manifest, f, indent=2)
        
    df_units_master.to_csv(STAGING_UNITS_DIR / 'units.csv', index=False)
    with open(STAGING_UNITS_DIR / '_units_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(units_manifest, f, indent=2)
        
    # 2. Final Production Masters
    df_barangs_master.to_csv(FINAL_BARANG_DIR / 'barangs.csv', index=False)
    df_lots_master.to_csv(FINAL_LOT_DIR / 'lots.csv', index=False)
    
    db_ordered_cols = [
        'id', 'number', 'legacy_number', 'lot_id', 'location_id',
        'status', 'condition', 'specification', 'price', 'image_url', 'vehicle_registration',
        'burden', 'project_id', 'created_at', 'updated_at', 'type',
        'classification', 'vendor_id'
    ]
    if 'specification' not in df_units_master.columns:
        df_units_master['specification'] = None
    df_units_final = df_units_master[[
        'id', 'number', 'legacy_number', 'lot_id', 'location_id',
        'status', 'condition', 'specification', 'price', 'image_url', 'burden',
        'project_id', 'created_at', 'updated_at', 'type',
        'classification', 'vendor_id'
    ]].copy()
    df_units_final['vehicle_registration'] = None
    df_units_final = df_units_final[db_ordered_cols]
    for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
        df_units_final[c] = pd.to_numeric(df_units_final[c], errors='coerce').astype('Int64')
    df_units_final.to_csv(FINAL_UNIT_DIR / 'units.csv', index=False)
    
    # 3. SQL Insert Scripts
    with open(FINAL_BARANG_DIR / 'barangs.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [barangs] ON;\nGO\n\n')
        for _, r in df_barangs_master.iterrows():
            f.write(f"INSERT INTO [barangs] ([id], [number], [subcategory_id], [brand_id], [uom_id], [name], [specification], [min_stock_threshold], [image_url], [last_restock_at], [created_at], [updated_at]) VALUES ({r['id']}, {escape_sql(r['number'])}, {r['subcategory_id']}, {r['brand_id']}, {r['uom_id']}, {escape_sql(r['name'])}, {escape_sql(r['specification'])}, {format_num_sql(r['min_stock_threshold'])}, {escape_sql(r['image_url'])}, {escape_sql(r['last_restock_at'])}, '{r['created_at']}', '{r['updated_at']}');\n")
        f.write('\nGO\nSET IDENTITY_INSERT [barangs] OFF;\nGO\n')
        
    with open(FINAL_LOT_DIR / 'lots.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [lots] ON;\nGO\n\n')
        for _, r in df_lots_master.iterrows():
            f.write(f"INSERT INTO [lots] ([id], [number], [barang_id], [organizer_id], [vendor_id], [legacy_vendor_id], [location_id], [initial_quantity], [current_quantity], [po_number], [date_of_receipt], [unit_price], [image_url], [burden], [project_id], [created_at], [updated_at]) VALUES ({r['id']}, {escape_sql(r['number'])}, {r['barang_id']}, {r['organizer_id']}, NULL, NULL, NULL, NULL, NULL, NULL, '{r['date_of_receipt']}', NULL, {escape_sql(r['image_url'])}, NULL, NULL, '{r['created_at']}', '{r['updated_at']}');\n")
        f.write('\nGO\nSET IDENTITY_INSERT [lots] OFF;\nGO\n')
        
    with open(FINAL_UNIT_DIR / 'units.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [units] ON;\nGO\n\n')
        for _, r in df_units_final.iterrows():
            f.write(f"INSERT INTO [units] ([id], [number], [legacy_number], [lot_id], [location_id], [status], [condition], [specification], [price], [image_url], [vehicle_registration], [burden], [project_id], [created_at], [updated_at], [type], [classification], [vendor_id]) VALUES ({r['id']}, {escape_sql(r['number'])}, {escape_sql(r['legacy_number'])}, {r['lot_id']}, {r['location_id']}, {escape_sql(r['status'])}, {escape_sql(r['condition'])}, {escape_sql(r['specification'])}, {format_num_sql(r['price'])}, {escape_sql(r['image_url'])}, {escape_sql(r['vehicle_registration'])}, {escape_sql(r['burden'])}, {format_num_sql(r['project_id'])}, '{r['created_at']}', '{r['updated_at']}', {escape_sql(r['type'])}, {escape_sql(r['classification'])}, {format_num_sql(r['vendor_id'])});\n")
        f.write('\nGO\nSET IDENTITY_INSERT [units] OFF;\nGO\n')
        
    # Global Quality Assertions
    assert len(df_units_master) in (6941, 6943), f'Expected 6,941 or 6,943 units, got {len(df_units_master)}'
    assert len(df_barangs_master) == len(df_lots_master)
    assert (df_barangs_master['id'].astype(int) == list(range(1, len(df_barangs_master) + 1))).all()
    assert (df_lots_master['id'].astype(int) == list(range(1, len(df_lots_master) + 1))).all()
    assert df_barangs_master['number'].is_unique
    assert df_lots_master['number'].is_unique
    assert df_barangs_master['number'].str.len().max() <= 14
    assert df_lots_master['number'].str.len().max() <= 26
    
    print(f' Master Sync Complete: {len(df_barangs_master):,} barangs, {len(df_lots_master):,} lots, {len(df_units_master):,} units.')
    return df_barangs_master, df_lots_master, df_units_master

print('Infrastructure setup and sync engine initialized successfully.')


Infrastructure setup and sync engine initialized successfully.


---
## Subcategory Consolidation: `ACC-AP` (Alat Pemadam Api Ringan / APAR)

### Business Context & Cleaning Rules:
In `ACC-AP`, asset entries originally contained location details typed directly into `spesifikasi` (*"Pos Security Samping"*, *"Depan Resepsionist"*, *"Samping Lift"*), decimal typing variations (`3,5` vs `3.5`), and redundant prefix tags (*"Jenis Apar"*, *"Apar Jenis"*).

**Consolidation Transformations Applied:**
1. **Strip Location Notes**: Physical locations belong in `units.location_id`, not in the catalog definition.
2. **Normalize Decimals**: Unify decimal comma `3,5` $\rightarrow$ `3.5`.
3. **Strip Prefixes**: Remove leading `Jenis Apar`, `Apar Jenis`, `Jenis`.
4. **Standardize Gases**: Standardize `CO 2` $\rightarrow$ `CO2`.
5. **Outcome**: Consolidates **32 fragmented items into exactly 12 standardized catalog barangs** across all 53 units.

In [2]:
# ---------------------------------------------------------
# Subcategory: ACC-AP Consolidation Cell
# ---------------------------------------------------------
SUB_CODE = 'ACC-AP'

def normalize_apar_spec(spec):
    if pd.isna(spec) or not str(spec).strip() or str(spec).strip().lower() in ('nan', 'none', 'null'):
        return None
    s = str(spec).replace('\r', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    
    # 1. Strip leading prefixes
    s = re.sub(r'^(jenis\s+apar\s+|apar\s+jenis\s+|jenis\s+)', '', s, flags=re.IGNORECASE)
    
    # 2. Normalize decimal commas to periods
    s = re.sub(r'(\d+),(\d+)', r'\1.\2', s)
    
    # 3. Standardize CO 2 -> CO2
    s = re.sub(r'co\s*2', 'CO2', s, flags=re.IGNORECASE)
    
    # 4. Extract core type and capacity, stripping location notes
    m_dc = re.search(r'dry\s+chemical\s*(\d+(?:\.\d+)?)\s*(?:kg)?', s, flags=re.IGNORECASE)
    if m_dc:
        return f'Dry Chemical {m_dc.group(1)} kg'
        
    m_co2 = re.search(r'co2\s*(\d+(?:\.\d+)?)\s*(?:kg)?', s, flags=re.IGNORECASE)
    if m_co2:
        return f'CO2 {m_co2.group(1)} kg'
        
    if 'powder' in s.lower() or 'abc' in s.lower():
        m_w = re.search(r'(\d+(?:\.\d+)?)\s*kg', s, flags=re.IGNORECASE)
        w = m_w.group(1) if m_w else '6'
        return f'Powder ABC {w} kg'
        
    return s

# Load partition datasets
df_ap_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_ap_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

df_ap_merged = df_ap_assets.merge(
    df_ap_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

# Group by (brand_id, name, normalized_spec)
groups = defaultdict(list)
for _, r in df_ap_merged.iterrows():
    b_id = resolve_brand_id(r.get('brand'))
    spec_norm = normalize_apar_spec(r.get('spesifikasi'))
    spec_key = spec_norm.upper() if spec_norm else ''
    groups[(b_id, 'APAR', spec_key)].append(r)

# Sort groups deterministically
sorted_groups = sorted(groups.items(), key=lambda item: int(float(item[1][0]['id'])))

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    display_spec = normalize_apar_spec(group_rows[0].get('spesifikasi'))
    
    # First non-null image
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': 'APAR',
        'specification': display_spec,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r['id']).strip()] = seq_counter
        
    seq_counter += 1

# Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

# Update units lot_id in staging
for idx in range(len(df_ap_units)):
    ams_id = str(df_ap_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_ap_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_ap_units.columns:
        df_ap_units[c] = pd.to_numeric(df_ap_units[c], errors='coerce').astype('Int64')
df_ap_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

# Synchronize global masters & SQL scripts
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# Local Assertions
assert len(df_sub_barangs) == 12, f'Expected 12 barangs, got {len(df_sub_barangs)}'
assert len(df_ap_units) == 53, f'Expected 53 units, got {len(df_ap_units)}'

print(f'\n[SUCCESS] Consolidated ACC-AP from 32 barangs down to {len(df_sub_barangs)} barangs across 53 units.')

# Display the consolidated items table
ap_display = df_sub_barangs[['id', 'number', 'brand_id', 'name', 'specification']].copy()
brand_id_to_name = dict(zip(df_brand['id'], df_brand['name']))
ap_display['brand'] = ap_display['brand_id'].astype(str).map(brand_id_to_name)
ap_unit_counts = df_ap_units.groupby('lot_id').size().to_dict()
ap_display['unit_count'] = ap_display['id'].astype(int).map(ap_unit_counts)
display(ap_display[['id', 'number', 'brand', 'name', 'specification', 'unit_count']])


 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-AP from 32 barangs down to 12 barangs across 53 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-AP-0001,Appron,APAR,Dry Chemical 3.5 kg,3
1,2,ACC-AP-0002,Firex,APAR,Dry Chemical 3.5 kg,15
2,3,ACC-AP-0003,Appron,APAR,Dry Chemical 2.5 kg,2
3,4,ACC-AP-0004,Firex,APAR,Dry Chemical 2.5 kg,1
4,5,ACC-AP-0005,Firex,APAR,Dry Chemical 3 kg,9
5,6,ACC-AP-0006,Firex,APAR,CO2 7 kg,2
6,7,ACC-AP-0007,Firex,APAR,Dry Chemical 6 kg,3
7,8,ACC-AP-0008,Firex,APAR,CO2 6 kg,2
8,9,ACC-AP-0009,Firex,APAR,CO2 5 kg,2
9,10,ACC-AP-0010,Firex,APAR,CO2 25 kg,1


---
## 2. Subcategory ACC-BT: Braket TV Consolidation
### Normalization Rules applied:
1. **Specification**: Ignore redundant 'Braket Besi' (all brackets are steel; canonical spec becomes None).
2. **Word Order**: Canonicalize name variations like 'Braket Standing TV' vs 'Braket TV Standing' into 'Braket TV Standing'.

In [3]:
# =====================================================================
# Subcategory ACC-BT: Braket TV Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-BT"
print(f"Consolidating {SUB_CODE}...")

def normalize_bt_name(raw_name):
    if not raw_name or pd.isna(raw_name):
        return "Braket TV Standing"
    words = set(re.findall(r"\b\w+\b", str(raw_name).strip().upper()))
    if {"BRAKET", "TV", "STANDING"}.issubset(words):
        return "Braket TV Standing"
    return str(raw_name).strip().title()

def normalize_bt_spec(raw_spec):
    if not raw_spec or pd.isna(raw_spec):
        return None
    s = str(raw_spec).strip()
    if s.upper() in ["BRAKET BESI", "BESI", "NULL", "NONE"]:
        return None
    return s if s else None

# 1. Load source partitions
df_bt_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_bt_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_bt_merged = df_bt_assets.merge(
    df_bt_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

# 2. Group by (brand_id, normalized_name, normalized_spec)
groups = defaultdict(list)
for _, r in df_bt_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    name_norm = normalize_bt_name(r.get("name"))
    spec_norm = normalize_bt_spec(r.get("spesifikasi"))
    spec_key = spec_norm.upper() if spec_norm else ""
    groups[(b_id, name_norm, spec_key)].append(r)

# Deterministic sorting
sorted_groups = sorted(groups.items(), key=lambda item: int(float(item[1][0]["id"])))

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = normalize_bt_spec(group_rows[0].get("spesifikasi"))
    
    # First non-null image
    first_img = None
    for r in group_rows:
        img = r.get("image_url") or r.get("image_url_unit")
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    updated_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))
    
    b_dict = {
        "id": seq_counter,
        "number": barang_num,
        "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id,
        "uom_id": 1,
        "name": name_val,
        "specification": display_spec,
        "min_stock_threshold": None,
        "image_url": first_img,
        "last_restock_at": None,
        "created_at": oldest_created,
        "updated_at": newest_updated
    }
    
    l_dict = {
        "id": seq_counter,
        "number": lot_num,
        "barang_id": seq_counter,
        "organizer_id": int(org_id),
        "vendor_id": None,
        "legacy_vendor_id": None,
        "location_id": None,
        "initial_quantity": None,
        "current_quantity": None,
        "po_number": None,
        "date_of_receipt": now_str,
        "unit_price": None,
        "image_url": first_img,
        "burden": None,
        "project_id": None,
        "created_at": now_str,
        "updated_at": now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r["id"]).strip()] = seq_counter
        
    seq_counter += 1

# 3. Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

# Update units lot_id in staging
for idx in range(len(df_bt_units)):
    ams_id = str(df_bt_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map:
        df_bt_units.at[idx, "lot_id"] = unit_lot_map[ams_id]

for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_bt_units.columns:
        df_bt_units[c] = pd.to_numeric(df_bt_units[c], errors="coerce").astype("Int64")
df_bt_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

# 4. Global synchronization
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# 5. Local assertions
assert len(df_sub_barangs) == 3, f"Expected 3 barangs for ACC-BT, got {len(df_sub_barangs)}"
assert len(df_bt_units) == 7, f"Expected 7 units for ACC-BT, got {len(df_bt_units)}"
print(f"\n[SUCCESS] Consolidated ACC-BT from 5 barangs down to {len(df_sub_barangs)} barangs across 7 units.")

# Display items summary table
bt_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
bt_display["brand"] = bt_display["brand_id"].astype(str).map(brand_id_to_name)
bt_unit_counts = df_bt_units.groupby("lot_id").size().to_dict()
bt_display["unit_count"] = bt_display["id"].astype(int).map(bt_unit_counts)
display(bt_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-BT...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-BT from 5 barangs down to 3 barangs across 7 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-BT-0001,Panasonic,Braket TV Standing,None,2
1,2,ACC-BT-0002,No Brand,Braket TV Standing,None,1
2,3,ACC-BT-0003,North Bayou,Braket TV Standing,None,4


---
## 3. Subcategory ACC-CM: Cermin Cembung Consolidation
### Normalization Rules applied:
1. Combine both items into one canonical item sharing Brand ID 123 (No Brand).
2. **Name**: 
3. **Specification**: 

In [4]:
# =====================================================================
# Subcategory ACC-CM: Cermin Cembung Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-CM"
print(f"Consolidating {SUB_CODE}...")

df_cm_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_cm_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_cm_merged = df_cm_assets.merge(
    df_cm_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

first_img = None
for _, r in df_cm_merged.iterrows():
    img = r.get("image_url") or r.get("image_url_unit")
    if pd.notna(img) and str(img).strip():
        first_img = str(img).strip()
        break

created_ats = [str(r.get("created_at") or r.get("created_at_unit")) for _, r in df_cm_merged.iterrows() if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
updated_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for _, r in df_cm_merged.iterrows() if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")
oldest_created = min(created_ats) if created_ats else now_str
newest_updated = max(updated_ats) if updated_ats else now_str

barang_num = f"{SUB_CODE}-0001"
lot_num = f"LOT-0001-26-{barang_num}"
org_id = resolve_organizer_id(df_cm_merged.iloc[0].get("organizer"))

b_dict = {
    "id": 1,
    "number": barang_num,
    "subcategory_id": sub_map[SUB_CODE],
    "brand_id": "123",
    "uom_id": 1,
    "name": "Cermin Cembung (Convex)",
    "specification": "Convex 45cm",
    "min_stock_threshold": None,
    "image_url": first_img,
    "last_restock_at": None,
    "created_at": oldest_created,
    "updated_at": newest_updated
}

l_dict = {
    "id": 1,
    "number": lot_num,
    "barang_id": 1,
    "organizer_id": int(org_id),
    "vendor_id": None,
    "legacy_vendor_id": None,
    "location_id": None,
    "initial_quantity": None,
    "current_quantity": None,
    "po_number": None,
    "date_of_receipt": now_str,
    "unit_price": None,
    "image_url": first_img,
    "burden": None,
    "project_id": None,
    "created_at": now_str,
    "updated_at": now_str
}

df_sub_barangs = pd.DataFrame([b_dict])
df_sub_lots = pd.DataFrame([l_dict])

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

df_cm_units["lot_id"] = 1
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_cm_units.columns:
        df_cm_units[c] = pd.to_numeric(df_cm_units[c], errors="coerce").astype("Int64")
df_cm_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 1, f"Expected 1 barang, got {len(df_sub_barangs)}"
assert len(df_cm_units) == 7, f"Expected 7 units, got {len(df_cm_units)}"
print(f"\n[SUCCESS] Consolidated ACC-CM from 2 barangs down to 1 barang across 7 units.")

cm_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
cm_display["brand"] = cm_display["brand_id"].astype(str).map(brand_id_to_name)
cm_display["unit_count"] = len(df_cm_units)
display(cm_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-CM...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-CM from 2 barangs down to 1 barang across 7 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-CM-0001,No Brand,Cermin Cembung (Convex),Convex 45cm,7


---
## 4. Subcategory ACC-DP: Dispenser Consolidation
### Normalization Rules applied:
1. Combine ACC-DP-0001 and ACC-DP-0002 into one entry using ACC-DP-0001 data (, , Miyako).
2. Convert ACC-DP-0003 name to Title Case (, Denpoo).

In [5]:
# =====================================================================
# Subcategory ACC-DP: Dispenser Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-DP"
print(f"Consolidating {SUB_CODE}...")

df_dp_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_dp_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_dp_merged = df_dp_assets.merge(
    df_dp_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

# Separate items: Group 1 (Miyako) and Group 2 (Denpoo)
# ams_asset_id 9419 is Denpoo (04565-ACC-DP-CFS-PTRE-24)
group1_rows = df_dp_merged[df_dp_merged["id"] != "9419"].copy()
group2_rows = df_dp_merged[df_dp_merged["id"] == "9419"].copy()

now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")
sub_barangs = []
sub_lots = []
unit_lot_map = {}

# Group 1: Miyako Dispenser Otomatis
first_img_1 = None
for _, r in group1_rows.iterrows():
    img = r.get("image_url") or r.get("image_url_unit")
    if pd.notna(img) and str(img).strip():
        first_img_1 = str(img).strip()
        break

c_ats_1 = [str(r.get("created_at") or r.get("created_at_unit")) for _, r in group1_rows.iterrows() if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
u_ats_1 = [str(r.get("updated_at") or r.get("updated_at_unit")) for _, r in group1_rows.iterrows() if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]

b1 = {
    "id": 1,
    "number": f"{SUB_CODE}-0001",
    "subcategory_id": sub_map[SUB_CODE],
    "brand_id": "101",
    "uom_id": 1,
    "name": "Dispenser",
    "specification": "Dispenser Otomatis",
    "min_stock_threshold": None,
    "image_url": first_img_1,
    "last_restock_at": None,
    "created_at": min(c_ats_1) if c_ats_1 else now_str,
    "updated_at": max(u_ats_1) if u_ats_1 else now_str
}
l1 = {
    "id": 1,
    "number": f"LOT-0001-26-{SUB_CODE}-0001",
    "barang_id": 1,
    "organizer_id": int(resolve_organizer_id(group1_rows.iloc[0].get("organizer"))),
    "vendor_id": None,
    "legacy_vendor_id": None,
    "location_id": None,
    "initial_quantity": None,
    "current_quantity": None,
    "po_number": None,
    "date_of_receipt": now_str,
    "unit_price": None,
    "image_url": first_img_1,
    "burden": None,
    "project_id": None,
    "created_at": now_str,
    "updated_at": now_str
}
sub_barangs.append(b1)
sub_lots.append(l1)
for _, r in group1_rows.iterrows():
    unit_lot_map[str(r["id"]).strip()] = 1

# Group 2: Denpoo Dispenser Kecil
first_img_2 = None
for _, r in group2_rows.iterrows():
    img = r.get("image_url") or r.get("image_url_unit")
    if pd.notna(img) and str(img).strip():
        first_img_2 = str(img).strip()
        break

c_ats_2 = [str(r.get("created_at") or r.get("created_at_unit")) for _, r in group2_rows.iterrows() if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
u_ats_2 = [str(r.get("updated_at") or r.get("updated_at_unit")) for _, r in group2_rows.iterrows() if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]

b2 = {
    "id": 2,
    "number": f"{SUB_CODE}-0002",
    "subcategory_id": sub_map[SUB_CODE],
    "brand_id": "143",
    "uom_id": 1,
    "name": "Dispenser Kecil",
    "specification": None,
    "min_stock_threshold": None,
    "image_url": first_img_2,
    "last_restock_at": None,
    "created_at": min(c_ats_2) if c_ats_2 else now_str,
    "updated_at": max(u_ats_2) if u_ats_2 else now_str
}
l2 = {
    "id": 2,
    "number": f"LOT-0001-26-{SUB_CODE}-0002",
    "barang_id": 2,
    "organizer_id": int(resolve_organizer_id(group2_rows.iloc[0].get("organizer"))),
    "vendor_id": None,
    "legacy_vendor_id": None,
    "location_id": None,
    "initial_quantity": None,
    "current_quantity": None,
    "po_number": None,
    "date_of_receipt": now_str,
    "unit_price": None,
    "image_url": first_img_2,
    "burden": None,
    "project_id": None,
    "created_at": now_str,
    "updated_at": now_str
}
sub_barangs.append(b2)
sub_lots.append(l2)
for _, r in group2_rows.iterrows():
    unit_lot_map[str(r["id"]).strip()] = 2

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

for idx in range(len(df_dp_units)):
    ams_id = str(df_dp_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map:
        df_dp_units.at[idx, "lot_id"] = unit_lot_map[ams_id]

for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_dp_units.columns:
        df_dp_units[c] = pd.to_numeric(df_dp_units[c], errors="coerce").astype("Int64")
df_dp_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 2, f"Expected 2 barangs, got {len(df_sub_barangs)}"
assert len(df_dp_units) == 9, f"Expected 9 units, got {len(df_dp_units)}"
print(f"\n[SUCCESS] Consolidated ACC-DP from 3 barangs down to 2 barangs across 9 units.")

dp_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
dp_display["brand"] = dp_display["brand_id"].astype(str).map(brand_id_to_name)
dp_unit_counts = df_dp_units.groupby("lot_id").size().to_dict()
dp_display["unit_count"] = dp_display["id"].astype(int).map(dp_unit_counts)
display(dp_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-DP...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-DP from 3 barangs down to 2 barangs across 9 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-DP-0001,Sanken,Dispenser,Dispenser Otomatis,8
1,2,ACC-DP-0002,Sonic,Dispenser Kecil,None,1


---
## 5. Subcategory ACC-GL: Glassboard Consolidation
### Normalization Rules applied:
1. **Standardize Typo**: Name normalized to  (Title Case).
2. **Specification Cleanup**: Strip non-size info (, ,  -> None) and embedded locations.
3. **Size Dimensions**: Canonicalized to  (Title Case).
4. **Grouping**: Group by .

In [6]:
# =====================================================================
# Subcategory ACC-GL: Glassboard Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-GL"
print(f"Consolidating {SUB_CODE}...")

def normalize_gl_spec(raw_spec):
    if not raw_spec or pd.isna(raw_spec):
        return None
    s = str(raw_spec).strip()
    m = re.search(r"(\d+\s*[xX]\s*\d+(?:\s*cm)?)", s)
    if m:
        dim = m.group(1).lower().replace("cm", "").strip()
        parts = re.split(r"\s*[xX]\s*", dim)
        if len(parts) == 2:
            return f"{parts[0].strip()} x {parts[1].strip()} cm"
        return m.group(1).strip()
    return None

df_gl_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_gl_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_gl_merged = df_gl_assets.merge(
    df_gl_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

groups = defaultdict(list)
for _, r in df_gl_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec_norm = normalize_gl_spec(r.get("spesifikasi"))
    spec_key = spec_norm if spec_norm else ""
    groups[(b_id, spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda item: int(float(item[1][0]["id"])))

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = spec_key if spec_key else None
    
    first_img = None
    for r in group_rows:
        img = r.get("image_url") or r.get("image_url_unit")
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    updated_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))
    
    b_dict = {
        "id": seq_counter,
        "number": barang_num,
        "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id,
        "uom_id": 1,
        "name": "Glassboard",
        "specification": display_spec,
        "min_stock_threshold": None,
        "image_url": first_img,
        "last_restock_at": None,
        "created_at": oldest_created,
        "updated_at": newest_updated
    }
    
    l_dict = {
        "id": seq_counter,
        "number": lot_num,
        "barang_id": seq_counter,
        "organizer_id": int(org_id),
        "vendor_id": None,
        "legacy_vendor_id": None,
        "location_id": None,
        "initial_quantity": None,
        "current_quantity": None,
        "po_number": None,
        "date_of_receipt": now_str,
        "unit_price": None,
        "image_url": first_img,
        "burden": None,
        "project_id": None,
        "created_at": now_str,
        "updated_at": now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r["id"]).strip()] = seq_counter
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

for idx in range(len(df_gl_units)):
    ams_id = str(df_gl_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map:
        df_gl_units.at[idx, "lot_id"] = unit_lot_map[ams_id]

for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_gl_units.columns:
        df_gl_units[c] = pd.to_numeric(df_gl_units[c], errors="coerce").astype("Int64")
df_gl_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 7, f"Expected 7 barangs, got {len(df_sub_barangs)}"
assert len(df_gl_units) == 54, f"Expected 54 units, got {len(df_gl_units)}"
print(f"\n[SUCCESS] Consolidated ACC-GL from 12 barangs down to 7 barangs across 54 units.")

gl_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
gl_display["brand"] = gl_display["brand_id"].astype(str).map(brand_id_to_name)
gl_unit_counts = df_gl_units.groupby("lot_id").size().to_dict()
gl_display["unit_count"] = gl_display["id"].astype(int).map(gl_unit_counts)
display(gl_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-GL...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-GL from 12 barangs down to 7 barangs across 54 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-GL-0001,Syly Art,Glassboard,90 x 80 cm,2
1,2,ACC-GL-0002,Syly Art,Glassboard,None,46
2,3,ACC-GL-0003,Syly Art,Glassboard,90 x 120 cm,1
3,4,ACC-GL-0004,Daya Kurnia,Glassboard,90 x 120 cm,2
4,5,ACC-GL-0005,Daya Kurnia,Glassboard,90 x 140 cm,1
5,6,ACC-GL-0006,Daya Kurnia,Glassboard,90 x 90 cm,1
6,7,ACC-GL-0007,No Brand,Glassboard,None,1


---
## 6. Formatting Polish: ACC-FT & ACC-GE (Title Case)
### Formatting Rules applied:
1. **ACC-FT**: Standardize name  -> .
2. **ACC-GE**: Standardize name  -> .

In [7]:
# =====================================================================
# Formatting Polish: ACC-FT & ACC-GE (Title Case Names)
# =====================================================================
# 1. Update ACC-FT
df_ft = pd.read_csv(STAGING_BARANGS_DIR / "ACC-FT.csv", dtype=str)
df_ft["name"] = df_ft["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_ft.to_csv(STAGING_BARANGS_DIR / "ACC-FT.csv", index=False)
print("ACC-FT names formatted to Title Case:")
display(df_ft[["number", "name"]])

# 2. Update ACC-GE
df_ge = pd.read_csv(STAGING_BARANGS_DIR / "ACC-GE.csv", dtype=str)
df_ge["name"] = df_ge["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_ge.to_csv(STAGING_BARANGS_DIR / "ACC-GE.csv", index=False)
print("ACC-GE names formatted to Title Case:")
display(df_ge[["number", "name"]])

# Synchronize global deliverables
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print(f"\n[SUCCESS] Formatted ACC-FT & ACC-GE to Title Case and synchronized masters.")


ACC-FT names formatted to Title Case:


,number,name
0,ACC-FT-0001,Foto


ACC-GE names formatted to Title Case:


,number,name
0,ACC-GE-0001,Gerobak Etalase
1,ACC-GE-0002,Gerobak Etalase Kantin


 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Formatted ACC-FT & ACC-GE to Title Case and synchronized masters.


---
## 7. Subcategory ACC-P3K: Kotak P3K Consolidation
### Normalization Rules applied:
1. Rename  ->  in specification.
2. Merge typo entry  (Kotoak P3K) into  (Kotak P3K).

In [8]:
# =====================================================================
# Subcategory ACC-P3K: Kotak P3K Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-P3K"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def norm_p3k_spec(s):
    if not s or pd.isna(s): return None
    val = str(s).strip()
    if val.upper() in ["STENLIST", "STENLIS", "STAINLESS"]:
        return "Stainless"
    return val

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec = norm_p3k_spec(r.get("spesifikasi"))
    spec_key = spec.upper() if spec else ""
    groups[(b_id, "Kotak P3K", spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = norm_p3k_spec(group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 4, f"Expected 4 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-P3K from 5 barangs down to 4 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-P3K...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-P3K from 5 barangs down to 4 barangs.


,id,number,brand_id,name,specification
0,1,ACC-P3K-0001,64,Kotak P3K,Stainless
1,2,ACC-P3K-0002,138,Kotak P3K,None
2,3,ACC-P3K-0003,64,Kotak P3K,Tinggi 45 x Lebar 30 Tebal 12 cm
3,4,ACC-P3K-0004,64,Kotak P3K,None


---
## 8. Subcategory ACC-PH: Pengharum Ruangan Consolidation
### Normalization Rules applied:
1. Strip brand names (Glade, Stella) from specifications.
2. Canonicalize name to  (standardizes ).
3. Combine 0006, 0007, 0010 (Stella, spec None) and 0005, 0008, 0009 (Glade, spec None).

In [9]:
# =====================================================================
# Subcategory ACC-PH: Pengharum Ruangan Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-PH"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def norm_ph_spec(s):
    if not s or pd.isna(s): return None
    val = str(s).strip()
    if val.upper() in ["GLADE", "STELLA", "NULL", "NONE", "NAN"]:
        return None
    return val

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec = norm_ph_spec(r.get("spesifikasi"))
    spec_key = spec.upper() if spec else ""
    groups[(b_id, "Pengharum Ruangan", spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = norm_ph_spec(group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 6, f"Expected 6 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-PH from 10 barangs down to 6 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-PH...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-PH from 10 barangs down to 6 barangs.


,id,number,brand_id,name,specification
0,1,ACC-PH-0001,120,Pengharum Ruangan,Pengharum Otomatis
1,2,ACC-PH-0002,175,Pengharum Ruangan,Warna Putih
2,3,ACC-PH-0003,120,Pengharum Ruangan,Pengarum Tempel
3,4,ACC-PH-0004,120,Pengharum Ruangan,Warna Putih
4,5,ACC-PH-0005,120,Pengharum Ruangan,None
5,6,ACC-PH-0006,175,Pengharum Ruangan,None


---
## 9. Subcategory ACC-PM: Perlengkapan Motor Consolidation
### Normalization Rules applied:
1. Strip personal names e.g. , .
2. Title Case names and specs.
3. Combine Helm into  (JS, ID 171) and Jas Hujan into  (Tiger Head, ID 172).

In [10]:
# =====================================================================
# Subcategory ACC-PM: Perlengkapan Motor Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-PM"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    n = str(r.get("name")).strip().upper()
    if "HELM" in n:
        groups[("171", "Helm Motor Op", "")].append(r)
    else:
        groups[("172", "Jas Hujan Motor Op", "")].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": None,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 2, f"Expected 2 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-PM from 5 barangs down to 2 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-PM...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-PM from 5 barangs down to 2 barangs.


,id,number,brand_id,name,specification
0,1,ACC-PM-0001,171,Helm Motor Op,None
1,2,ACC-PM-0002,172,Jas Hujan Motor Op,None


---
## 10. Subcategory ACC-POT: Pot Tanaman Consolidation
### Normalization Rules applied:
1. Standardize names and specs to Title Case.
2. Combine  (POT) into  (Pot Tanaman, spec None).

In [11]:
# =====================================================================
# Subcategory ACC-POT: Pot Tanaman Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-POT"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def norm_pot_spec(s):
    if not s or pd.isna(s) or str(s).strip().upper() in ["NAN", "NULL", "NONE"]: return None
    val = str(s).strip()
    if "TANAH LIAT" in val.upper(): return "Tanah Liat Coklat"
    if "WARNA COKLAT" in val.upper(): return "Warna Coklat"
    if "MATI" in val.upper(): return "Tanaman Mati"
    return val.title()

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec = norm_pot_spec(r.get("spesifikasi"))
    spec_key = spec.upper() if spec else ""
    groups[(b_id, "Pot Tanaman", spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = norm_pot_spec(group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 4, f"Expected 4 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-POT from 5 barangs down to 4 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-POT...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-POT from 5 barangs down to 4 barangs.


,id,number,brand_id,name,specification
0,1,ACC-POT-0001,123,Pot Tanaman,Warna Coklat
1,2,ACC-POT-0002,123,Pot Tanaman,None
2,3,ACC-POT-0003,123,Pot Tanaman,Tanah Liat Coklat
3,4,ACC-POT-0004,123,Pot Tanaman,Tanaman Mati


---
## 11. Subcategories ACC-TAP & ACC-TK Consolidation
### Normalization Rules applied:
1. **ACC-TAP**: Rename  -> .
2. **ACC-TK**: Rename  -> , and fix  ->  in specs.

In [12]:
# =====================================================================
# Subcategories ACC-TAP & ACC-TK Renaming & Polish
# =====================================================================
# 1. ACC-TAP
df_tap = pd.read_csv(STAGING_BARANGS_DIR / "ACC-TAP.csv", dtype=str)
df_tap["name"] = df_tap["name"].replace({"Termos Stenlis": "Termos Stainless"})
df_tap.to_csv(STAGING_BARANGS_DIR / "ACC-TAP.csv", index=False)

# 2. ACC-TK
df_tk = pd.read_csv(STAGING_BARANGS_DIR / "ACC-TK.csv", dtype=str)
df_tk["name"] = df_tk["name"].replace({"Troly Stenlis Pantry": "Troli Stainless Pantry"})
df_tk["specification"] = df_tk["specification"].replace({
    "Stenlis": "Stainless",
    "Troli Stenlis Lt 1": "Troli Stainless Lt 1"
})
df_tk.to_csv(STAGING_BARANGS_DIR / "ACC-TK.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print("\n[SUCCESS] Updated ACC-TAP and ACC-TK.")
display(df_tap[["number", "name", "specification"]])
display(df_tk[["number", "name", "specification"]])


 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Updated ACC-TAP and ACC-TK.


,number,name,specification
0,ACC-TAP-0001,Termos Stainless,5 Liter
1,ACC-TAP-0002,Termos Air Panas,NaN
2,ACC-TAP-0003,Termos Air Panas,NaN
3,ACC-TAP-0004,Termos Air Panas,NaN


,number,name,specification
0,ACC-TK-0001,Troli Stainless Pantry,Stainless
1,ACC-TK-0002,Troli Kecil,NaN
2,ACC-TK-0003,Troli Kecil,Troli Stainless Lt 1


---
## 12. Subcategory ACC-TS: Tempat Sampah Consolidation
### Normalization Rules applied:
1. Deprecate  -> canonical .
2. Strip  (and typos) and  from name; move  from name to specification.
3. Standardize stainless spec to .
4. Maintain distinct sizes (, , ).

In [13]:
# =====================================================================
# Subcategory ACC-TS: Tempat Sampah Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-TS"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def clean_ts_entry(raw_name, raw_spec):
    n = str(raw_name).strip() if pd.notna(raw_name) else "Tempat Sampah"
    s = str(raw_spec).strip() if pd.notna(raw_spec) else ""
    has_besi = "besi" in n.lower()
    n = re.sub(r"(?i)\btong\s+sampah\b", "Tempat Sampah", n)
    n = re.sub(r"(?i)\b(stenlis|stenlist|stanless|stanlees|srenlist|stainless|plastik|besi)\b", "", n)
    n = re.sub(r"\s+", " ", n).strip()
    if not n: n = "Tempat Sampah"
    n = n.title()
    
    is_stainless = bool(re.search(r"(?i)\b(stenlis|stenlist|stanless|stanlees|srenlist|stainless)\b", s))
    if is_stainless:
        s_clean = "Stainless"
    elif has_besi or "besi" in s.lower():
        s_clean = "Besi"
    elif "bundar" in s.lower():
        s_clean = "Bundar"
    elif "warna merah" in s.lower():
        s_clean = "Warna Merah"
    elif "ts-doble" in s.lower():
        s_clean = "TS-Doble"
    elif "12 liter" in s.lower():
        s_clean = "Plastik Tutup 12 Liter"
    elif "10 liter" in s.lower():
        s_clean = "Plastik Tutup 10 Liter"
    elif "60x30" in s.lower():
        s_clean = "Ukuran 60 x 30 cm"
    elif s == "" or s.lower() in ["nan", "null", "none"]:
        s_clean = None
    else:
        s_clean = s.title()
    return n, s_clean

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    c_name, c_spec = clean_ts_entry(r.get("name"), r.get("spesifikasi"))
    spec_key = c_spec.upper() if c_spec else ""
    groups[(b_id, c_name, spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    _, display_spec = clean_ts_entry(group_rows[0].get("name"), group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 17, f"Expected 17 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-TS from 25 barangs down to 17 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-TS...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-TS from 25 barangs down to 17 barangs.


,id,number,brand_id,name,specification
0,1,ACC-TS-0001,125,Tempat Sampah,Bundar
1,2,ACC-TS-0002,79,Tempat Sampah,Stainless
2,3,ACC-TS-0003,125,Tempat Sampah,Stainless
3,4,ACC-TS-0004,79,Tempat Sampah,None
4,5,ACC-TS-0005,64,Tempat Sampah,None
5,6,ACC-TS-0006,79,Tempat Sampah,Bundar
6,7,ACC-TS-0007,64,Tempat Sampah,TS-Doble
7,8,ACC-TS-0008,79,Tempat Sampah B3,Warna Merah
8,9,ACC-TS-0009,64,Tempat Sampah,Plastik Tutup 12 Liter
9,10,ACC-TS-0010,64,Tempat Sampah,Plastik Tutup 10 Liter


---
## 13. Subcategory ACC-WB: Whiteboard Consolidation
### Normalization Rules applied:
1. Combine  (White Board) and  (Whiteboard) into canonical  (No Brand).

In [14]:
# =====================================================================
# Subcategory ACC-WB: Whiteboard Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-WB"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    n = "Panaboard" if b_id == "25" else "Whiteboard"
    groups[(b_id, n, "")].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": None,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 2, f"Expected 2 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-WB from 3 barangs down to 2 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-WB...
 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ACC-WB from 3 barangs down to 2 barangs.


,id,number,brand_id,name,specification
0,1,ACC-WB-0001,123,Whiteboard,None
1,2,ACC-WB-0002,25,Panaboard,None


---
## 14. Formatting Polish: ACC-PK & ACC-STB (Title Case)
### Formatting Rules applied:
1. **ACC-PK**: Name  -> .
2. **ACC-STB**: Name  -> .

In [15]:
# =====================================================================
# Formatting Polish: ACC-PK & ACC-STB (Title Case Names)
# =====================================================================
df_pk = pd.read_csv(STAGING_BARANGS_DIR / "ACC-PK.csv", dtype=str)
df_pk["name"] = df_pk["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_pk.to_csv(STAGING_BARANGS_DIR / "ACC-PK.csv", index=False)

df_stb = pd.read_csv(STAGING_BARANGS_DIR / "ACC-STB.csv", dtype=str)
df_stb["name"] = df_stb["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_stb.to_csv(STAGING_BARANGS_DIR / "ACC-STB.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print("\n[SUCCESS] Formatted ACC-PK and ACC-STB to Title Case.")
display(df_pk[["number", "name"]])
display(df_stb[["number", "name"]])


 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Formatted ACC-PK and ACC-STB to Title Case.


,number,name
0,ACC-PK-0001,Pot Kecil


,number,name
0,ACC-STB-0001,Scaffolding
1,ACC-STB-0002,Storage Box


---
## 15. Subcategory ACC-GPS: GPS Consolidation
### Normalization Rules applied:
1. Remove  from  name ( -> ) as it is redundant with brand (DJI, Brand ID 90).

In [16]:
# =====================================================================
# Subcategory ACC-GPS: Remove Redundant Brand Name
# =====================================================================
df_gps = pd.read_csv(STAGING_BARANGS_DIR / "ACC-GPS.csv", dtype=str)
df_gps["name"] = df_gps["name"].replace({"DJI D-RTK 3": "D-RTK 3"})
df_gps.to_csv(STAGING_BARANGS_DIR / "ACC-GPS.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print("\n[SUCCESS] Updated ACC-GPS-0001 name to D-RTK 3.")
display(df_gps[["number", "brand_id", "name", "specification"]])


 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Updated ACC-GPS-0001 name to D-RTK 3.


,number,brand_id,name,specification
0,ACC-GPS-0001,90,D-RTK 3,Multifunctional Station + Tripod Station


---
## 16. Subcategory COMP-DESK: Computer Desktop Consolidation
### Normalization Rules applied:
1. **Unit-Level Specifications**: In accordance with Smart app inventory migrations, hardware specifications are moved to `units.specification`, leaving `barangs.specification` as `NULL`.
2. **Brand & Device Name Separation**: Redundant brand names and hardware specifications in titles are omitted.
3. **Designated First Barang**: `PC 3D` is designated as `COMP-DESK-0001`.
4. **Consolidated Device Names (23 Barangs)**:
   - **Rakitan**: `PC 3D` (0001), `P4` (0002), `AMD Athlon` (0003), `PC Lainnya (Lawas)` (0004).
   - **Compaq**: `P3` (0005), `P4` (0006).
   - **Dell**: `P4` (0007), `Inspiron 530` (0008), `Vostro` (0009 - merges 3670, 3668, 360, VOSTRO, and 0139 Dell 19.5).
   - **Lenovo**: `A70Z` (0010), `IdeaCentre B340` (0011), `IdeaCentre C440` (0012), `K330B` (0013), `Edge 71` (0014).
   - **Acer**: `C22-1700` (0015).
   - **HP**: `P4` (0016), `Omni` (0017), `MS 200` (0018), `280 G3` (0019), `ProOne 400 G2` (0020), `Slimline 400-020L` (0021), `20 All-in-One` (0022), `Pavilion` (0023).


In [17]:
# =====================================================================
# Subcategory COMP-DESK: Computer Desktop Consolidation
# =====================================================================
SUB_CODE = 'COMP-DESK'

df_desk_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_desk_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

# Clean and attach specification to units
def clean_unit_spec(val):
    if pd.isna(val) or val is None:
        return None
    s = str(val).replace('\r', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s and s.lower() not in ('nan', 'none', 'null') else None

asset_spec_map = {str(r['id']).strip(): clean_unit_spec(r.get('spesifikasi')) for _, r in df_desk_assets.iterrows()}
df_desk_units['specification'] = df_desk_units['ams_asset_id'].astype(str).map(asset_spec_map)

# Ordered consolidation targets: COMP-DESK-0001 through COMP-DESK-0023
TARGET_ORDER = [
    ('5', 'PC 3D'),       # COMP-DESK-0001 (Rakitan)
    ('5', 'P4'),                  # COMP-DESK-0002 (Rakitan)
    ('5', 'AMD Athlon'),          # COMP-DESK-0003 (Rakitan)
    ('5', 'PC Lainnya (Lawas)'),  # COMP-DESK-0004 (Rakitan)
    ('1', 'P3'),                  # COMP-DESK-0005 (Compaq)
    ('1', 'P4'),                  # COMP-DESK-0006 (Compaq)
    ('6', 'P4'),                  # COMP-DESK-0007 (Dell)
    ('6', 'Inspiron 530'),        # COMP-DESK-0008 (Dell)
    ('6', 'Vostro'),              # COMP-DESK-0009 (Dell)
    ('46', 'A70Z'),               # COMP-DESK-0010 (Lenovo)
    ('46', 'IdeaCentre B340'),    # COMP-DESK-0011 (Lenovo)
    ('46', 'IdeaCentre C440'),    # COMP-DESK-0012 (Lenovo)
    ('46', 'K330B'),              # COMP-DESK-0013 (Lenovo)
    ('46', 'Edge 71'),            # COMP-DESK-0014 (Lenovo)
    ('27', 'C22-1700'),           # COMP-DESK-0015 (Acer)
    ('9', 'P4'),                  # COMP-DESK-0016 (HP)
    ('9', 'Omni'),                # COMP-DESK-0017 (HP)
    ('9', 'MS 200'),              # COMP-DESK-0018 (HP)
    ('9', '280 G3'),              # COMP-DESK-0019 (HP)
    ('9', 'ProOne 400 G2'),       # COMP-DESK-0020 (HP)
    ('9', 'Slimline 400-020L'),   # COMP-DESK-0021 (HP)
    ('9', '20 All-in-One'),       # COMP-DESK-0022 (HP)
    ('9', 'Pavilion')             # COMP-DESK-0023 (HP)
]

def map_desk_asset(row):
    b = str(row.get('brand', '')).strip().upper()
    n = str(row.get('name', '')).strip()
    a_id = str(row.get('id', '')).strip()
    
    if b == 'COMPAQ':
        if 'P3:' in n: return ('1', 'P3')
        if 'P4:' in n: return ('1', 'P4')
    elif b == 'DELL':
        if 'P4:' in n: return ('6', 'P4')
        if 'INSPIRON' in n.upper(): return ('6', 'Inspiron 530')
        if 'VOSTRO' in n.upper() or '19.5' in n or a_id == '2744': return ('6', 'Vostro')
    elif b == 'LENOVO':
        if 'A70Z' in n.upper(): return ('46', 'A70Z')
        if 'B340' in n.upper(): return ('46', 'IdeaCentre B340')
        if 'C440' in n.upper(): return ('46', 'IdeaCentre C440')
        if 'K330B' in n.upper(): return ('46', 'K330B')
        if 'EDGE 71' in n.upper(): return ('46', 'Edge 71')
    elif b == 'ACER':
        if 'C22-1700' in n.upper(): return ('27', 'C22-1700')
    elif b == 'HP':
        if 'P4:' in n: return ('9', 'P4')
        if 'OMNI' in n.upper(): return ('9', 'Omni')
        if 'MS 200' in n.upper(): return ('9', 'MS 200')
        if '280 G3' in n.upper(): return ('9', '280 G3')
        if 'PRO ONE 400' in n.upper(): return ('9', 'ProOne 400 G2')
        if 'SLIMLINE' in n.upper(): return ('9', 'Slimline 400-020L')
        if any(k in n.upper() for k in ['20 A 210', '20 B 110', 'ALL IN ONE PC']): return ('9', '20 All-in-One')
        if any(k in n.upper() for k in ['PAVILON', 'P6-', 'P 6', 'P7-', 'P 500-', 'P 550-']): return ('9', 'Pavilion')
    elif b == 'RAKITAN':
        if 'P4:' in n: return ('5', 'P4')
        if 'ATHLON' in n.upper(): return ('5', 'AMD Athlon')
        if any(k in n.upper() for k in ['3D', 'PIPING', 'CIVIL']) or a_id in ['11855', '7888']:
            return ('5', 'PC 3D')
        return ('5', 'PC Lainnya (Lawas)')
        
    raise ValueError(f'Unmapped asset: id={a_id}, brand={b}, name={n}')

df_desk_merged = df_desk_assets.merge(
    df_desk_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_desk_merged.iterrows():
    target_key = map_desk_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val) in TARGET_ORDER:
    group_rows = groups.get((b_id, name_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': None,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r['id']).strip()] = seq_counter
        
    seq_counter += 1

# Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

# Update units lot_id and specification in staging
for idx in range(len(df_desk_units)):
    ams_id = str(df_desk_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_desk_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_desk_units.columns:
        df_desk_units[c] = pd.to_numeric(df_desk_units[c], errors='coerce').astype('Int64')
df_desk_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

# Synchronize global masters & SQL scripts
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# Local Assertions
assert len(df_sub_barangs) == 23, f'Expected 23 barangs, got {len(df_sub_barangs)}'
assert len(df_desk_units) == 782, f'Expected 782 units, got {len(df_desk_units)}'
assert df_sub_barangs.at[0, 'name'] == 'PC 3D', 'First barang must be PC 3D'
assert df_sub_barangs.at[0, 'number'] == 'COMP-DESK-0001', 'First barang number must be COMP-DESK-0001'

print(f'\n[SUCCESS] Consolidated COMP-DESK from 152 barangs down to {len(df_sub_barangs)} barangs across 782 units.')


 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated COMP-DESK from 152 barangs down to 23 barangs across 782 units.


---
## 17. Subcategory COMP-NB: Laptop / Notebook Consolidation
### Granular Brand & Ongoing Series Normalization

**Category Rules & Approach:**
1. **Bloat Reduction**: Consolidates 263 bloated single-procurement records into 55 clean, series-based master barangs.
2. **Specification Preservation**: All hardware specifications from AMS are migrated directly to `units.specification`. Master `barangs.specification` is set to `NULL`.
3. **Series-Based Naming**: Preserves active/ongoing brand series (e.g. `Pavilion`, `Inspiron`, `IdeaPad Slim`, `Vivobook K`), allowing users to procure the latest models without creating redundant barang masters.
4. **Admin Discrepancy Fixes**: Corrects misbranded cross-brand assets (MSI Thin GF63, HP Omen 15, Asus Vivobook 14X) and legacy typos.

In [18]:
# =============================================================================
# Subcategory COMP-NB (Laptop / Notebook) Consolidation
# =============================================================================
SUB_CODE = 'COMP-NB'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_nb_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_nb_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

# Clean and attach specification to units
def clean_unit_spec(val):
    if pd.isna(val) or val is None:
        return None
    s = str(val).replace('\r', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s and s.lower() not in ('nan', 'none', 'null') else None

asset_spec_map = {str(r['id']).strip(): clean_unit_spec(r.get('spesifikasi')) for _, r in df_nb_assets.iterrows()}
df_nb_units['specification'] = df_nb_units['ams_asset_id'].astype(str).map(asset_spec_map)

# Ordered consolidation targets: COMP-NB-0001 through COMP-NB-0055
TARGET_ORDER = [
    ('7', 'P3'),
    ('15', 'VAIO'),
    ('9', 'P4'),
    ('27', 'Turion'),
    ('31', 'S425'),
    ('1', 'Presario'),
    ('6', 'Inspiron'),
    ('7', 'Satellite'),
    ('27', 'Ultrabook'),
    ('49', 'A'),
    ('51', 'MacBook Pro'),
    ('9', 'EliteBook'),
    ('51', 'MacBook Air'),
    ('49', 'N'),
    ('46', 'Core 2 Duo'),
    ('27', 'Core 2 Duo'),
    ('49', 'X'),
    ('9', 'Omen'),
    ('9', 'Pavilion'),
    ('9', 'ProBook'),
    ('49', 'ROG'),
    ('46', 'ThinkPad'),
    ('9', 'Spectre'),
    ('6', 'XPS'),
    ('9', '240 G6'),
    ('9', 'Envy'),
    ('9', 'Laptop 14s'),
    ('49', 'TUF Gaming'),
    ('49', 'Zenbook'),
    ('49', 'P'),
    ('6', 'Vostro'),
    ('27', 'Aspire'),
    ('46', 'IdeaPad S'),
    ('46', 'IdeaPad Slim'),
    ('46', 'Yoga Slim'),
    ('46', 'ThinkBook'),
    ('46', 'Yoga'),
    ('46', 'Legion'),
    ('46', 'IdeaPad Gaming'),
    ('49', 'Vivobook K'),
    ('27', 'Swift'),
    ('27', 'Nitro'),
    ('49', 'Vivobook'),
    ('46', 'IdeaPad Flex'),
    ('49', 'Vivobook S'),
    ('49', 'Vivobook Pro'),
    ('31', 'Thin'),
    ('49', 'Vivobook X'),
    ('46', 'LOQ'),
    ('46', 'V14'),
    ('46', 'IdeaPad Duet'),
    ('31', 'Cyborg'),
    ('9', 'Victus'),
    ('27', 'Swift Go'),
    ('49', 'ExpertBook')
]

def map_nb_asset(row):
    a_id = str(row.get('id', '')).strip()
    b = str(row.get('brand', '')).strip().upper()
    n = str(row.get('name', '')).strip()
    nl = n.lower()
    spec = str(row.get('spesifikasi', '')).lower()
    
    # Discrepancy corrections
    if a_id == '8070': return ('31', 'Thin')
    if a_id == '7591': return ('9', 'Omen')
    if a_id == '8643': return ('49', 'Vivobook X')
    if a_id == '5170': return ('6', 'Inspiron')
    if a_id == '5174': return ('9', 'Pavilion')
    if a_id == '10718': return ('46', 'IdeaPad Slim')
    if a_id == '3476': return ('7', 'Satellite')
    
    # 1. Apple (51)
    if b == 'APPLE':
        if 'pro' in nl: return ('51', 'MacBook Pro')
        return ('51', 'MacBook Air')
        
    # 2. Compaq (1)
    if b == 'COMPAQ':
        return ('1', 'Presario')
        
    # 3. Sony (15)
    if b == 'SONY':
        return ('15', 'VAIO')
        
    # 4. Toshiba (7)
    if b == 'TOSHIBA':
        if 'p3' in nl: return ('7', 'P3')
        return ('7', 'Satellite')
        
    # 5. MSI (31)
    if b == 'MSI':
        if 's425' in nl: return ('31', 'S425')
        if 'cyborg' in nl: return ('31', 'Cyborg')
        return ('31', 'Thin')
        
    # 6. Dell (6)
    if b == 'DELL':
        if 'xps' in nl: return ('6', 'XPS')
        if 'v3480' in nl or 'vostro' in nl: return ('6', 'Vostro')
        return ('6', 'Inspiron')
        
    # 7. Acer (27)
    if b == 'ACER':
        if 'turion' in nl: return ('27', 'Turion')
        if 'core 2 duo' in spec or n == 'Acer': return ('27', 'Core 2 Duo')
        if 'ultrabook' in nl: return ('27', 'Ultrabook')
        if 'nitro' in nl: return ('27', 'Nitro')
        if 'swift go' in nl or 'ag 14' in nl: return ('27', 'Swift Go')
        if 'swift' in nl: return ('27', 'Swift')
        return ('27', 'Aspire')
        
    # 8. HP (9)
    if b == 'HP':
        if 'p4' in nl: return ('9', 'P4')
        if 'elitebook' in nl: return ('9', 'EliteBook')
        if 'spectre' in nl: return ('9', 'Spectre')
        if 'victus' in nl: return ('9', 'Victus')
        if 'omen' in nl: return ('9', 'Omen')
        if 'envy' in nl: return ('9', 'Envy')
        if 'probook' in nl or 'pro book' in nl: return ('9', 'ProBook')
        if '240 g6' in nl: return ('9', '240 G6')
        if 'pavilion' in nl or '14-bs' in nl: return ('9', 'Pavilion')
        return ('9', 'Laptop 14s')
        
    # 9. Asus (49)
    if b in ('ASUS', 'ASUS '):
        if 'tuf' in nl or 'fx50' in nl or 'fa50' in nl: return ('49', 'TUF Gaming')
        if 'rog' in nl or 'zephyrus' in nl: return ('49', 'ROG')
        if 'expertbook' in nl: return ('49', 'ExpertBook')
        if 'zenbook' in nl: return ('49', 'Zenbook')
        # Vivobooks:
        if 'pro' in nl and 'vivo' in nl: return ('49', 'Vivobook Pro')
        if ('s14' in nl or 's 14' in nl) and 'vivo' in nl: return ('49', 'Vivobook S')
        if '14x' in nl and 'vivo' in nl: return ('49', 'Vivobook X')
        if 'k413' in nl or 'k513' in nl or 'a413' in nl: return ('49', 'Vivobook K')
        if 'vivo' in nl: return ('49', 'Vivobook')
        # Legacy A, X, N, P:
        if any(k in nl for k in ['a43', 'a45', 'a46', 'a 46', 'a516', 'core i3']): return ('49', 'A')
        if 'x450' in nl: return ('49', 'X')
        if 'n 550' in nl or 'n550' in nl: return ('49', 'N')
        if 'p1440' in nl: return ('49', 'P')
        return ('49', 'A')
        
    # 10. Lenovo (46)
    if b == 'LENOVO':
        if n == 'Lenovo': return ('46', 'Core 2 Duo')
        if 'thinkpad' in nl: return ('46', 'ThinkPad')
        if 'thinkbook' in nl: return ('46', 'ThinkBook')
        if 'v14' in nl: return ('46', 'V14')
        if 'loq' in nl: return ('46', 'LOQ')
        if 'legion' in nl: return ('46', 'Legion')
        if 'yoga slim' in nl or 'yogaslim' in nl: return ('46', 'Yoga Slim')
        if 'yoga' in nl: return ('46', 'Yoga')
        # Ideapads:
        if 'gaming' in nl: return ('46', 'IdeaPad Gaming')
        if 'flex' in nl: return ('46', 'IdeaPad Flex')
        if 'duet' in nl: return ('46', 'IdeaPad Duet')
        if any(k in nl for k in ['slim', 'ip 3', 'ip 5', 'ideapad 3']): return ('46', 'IdeaPad Slim')
        if 's145' in nl: return ('46', 'IdeaPad S')
        return ('46', 'IdeaPad Slim')

    raise ValueError(f'Unmapped asset: id={a_id}, brand={b}, name={n}')

df_nb_merged = df_nb_assets.merge(
    df_nb_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_nb_merged.iterrows():
    target_key = map_nb_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val) in TARGET_ORDER:
    group_rows = groups.get((b_id, name_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': None,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r['id']).strip()] = seq_counter
        
    seq_counter += 1

# Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

# Update units lot_id and specification in staging
for idx in range(len(df_nb_units)):
    ams_id = str(df_nb_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_nb_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_nb_units.columns:
        df_nb_units[c] = pd.to_numeric(df_nb_units[c], errors='coerce').astype('Int64')
df_nb_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

# Synchronize global masters & SQL scripts
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# Local Assertions
assert len(df_sub_barangs) == 55, f'Expected 55 barangs, got {len(df_sub_barangs)}'
assert len(df_nb_units) == 607, f'Expected 607 units, got {len(df_nb_units)}'
assert df_sub_barangs.at[0, 'name'] == 'P3', 'First barang must be P3'
assert df_sub_barangs.at[0, 'number'] == 'COMP-NB-0001', 'First barang number must be COMP-NB-0001'

print(f"\n[SUCCESS] Consolidated COMP-NB from 263 barangs down to {len(df_sub_barangs)} barangs across 607 units.")



Processing Subcategory: COMP-NB

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated COMP-NB from 263 barangs down to 55 barangs across 607 units.


---
## 18. Subcategory COMP-SVR: Server Consolidation
### Series-Level Deduplication & SAN Storage Merging

**Category Rules & Approach:**
1. **Bloat Reduction**: Consolidates 14 single-procurement records into 6 clean, series-based master barangs.
2. **Specification Migration**: All hardware configurations from AMS are preserved directly in `units.specification`. Master `barangs.specification` is set to `NULL`.
3. **Series Normalization**: Merges ProLiant rack/tower models into `ProLiant`, MSA storage into `MSA SAN Storage`, and QNAP NAS models into `TVS` and `TS`.

In [19]:
# =============================================================================
# Subcategory COMP-SVR (Server) Consolidation
# =============================================================================
SUB_CODE = 'COMP-SVR'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_svr_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_svr_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

# Clean and attach specification to units
def clean_unit_spec(val):
    if pd.isna(val) or val is None:
        return None
    s = str(val).replace('\r', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s and s.lower() not in ('nan', 'none', 'null') else None

asset_spec_map = {str(r['id']).strip(): clean_unit_spec(r.get('spesifikasi')) for _, r in df_svr_assets.iterrows()}
df_svr_units['specification'] = df_svr_units['ams_asset_id'].astype(str).map(asset_spec_map)

# Ordered consolidation targets: COMP-SVR-0001 through COMP-SVR-0006
TARGET_ORDER = [
    ('6', 'P3'),
    ('5', 'Xeon'),
    ('9', 'ProLiant'),
    ('9', 'MSA SAN Storage'),
    ('70', 'TVS'),
    ('70', 'TS')
]

def map_svr_asset(row):
    b = str(row.get('brand', '')).strip().upper()
    n = str(row.get('name', '')).strip().upper()
    
    if b == 'DELL':
        return ('6', 'P3')
    if b == 'RAKITAN':
        return ('5', 'Xeon')
    if b == 'HP':
        if 'MSA' in n:
            return ('9', 'MSA SAN Storage')
        return ('9', 'ProLiant')
    if b == 'QNAP':
        if 'TVS' in n:
            return ('70', 'TVS')
        if 'TS' in n:
            return ('70', 'TS')
        return ('70', 'NAS')
        
    raise ValueError(f"Unmapped asset: id={row.get('id')}, brand={b}, name={n}")

df_svr_merged = df_svr_assets.merge(
    df_svr_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_svr_merged.iterrows():
    target_key = map_svr_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val) in TARGET_ORDER:
    group_rows = groups.get((b_id, name_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': None,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r['id']).strip()] = seq_counter
        
    seq_counter += 1

# Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

# Update units lot_id and specification in staging
for idx in range(len(df_svr_units)):
    ams_id = str(df_svr_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_svr_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_svr_units.columns:
        df_svr_units[c] = pd.to_numeric(df_svr_units[c], errors='coerce').astype('Int64')
df_svr_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

# Synchronize global masters & SQL scripts
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# Local Assertions
assert len(df_sub_barangs) == 6, f'Expected 6 barangs, got {len(df_sub_barangs)}'
assert len(df_svr_units) == 17, f'Expected 17 units, got {len(df_svr_units)}'
assert df_sub_barangs.at[0, 'name'] == 'P3', 'First barang must be P3'
assert df_sub_barangs.at[0, 'number'] == 'COMP-SVR-0001', 'First barang number must be COMP-SVR-0001'

print(f"\n[SUCCESS] Consolidated COMP-SVR from 14 barangs down to {len(df_sub_barangs)} barangs across 17 units.")



Processing Subcategory: COMP-SVR

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated COMP-SVR from 14 barangs down to 6 barangs across 17 units.


---
## 19. Subcategory COMP-WORK: Workstation Consolidation
### Workstation Series Consolidation & Cross-Brand Discrepancy Fix

**Category Rules & Approach:**
1. **Bloat Reduction**: Consolidates 9 bloated records into 2 clean master series: HP `Z` and Dell `Precision Tower`.
2. **Specification Migration**: All workstation hardware specs are preserved in `units.specification`. Master `barangs.specification` is set to `NULL`.
3. **Cross-Brand Correction**: Reassigns misbranded asset 6335 (HP Z6 G4 entered under Dell) to Brand HP.

In [20]:
# =============================================================================
# Subcategory COMP-WORK (Workstation) Consolidation
# =============================================================================
SUB_CODE = 'COMP-WORK'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_wrk_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_wrk_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

# Clean and attach specification to units
def clean_unit_spec(val):
    if pd.isna(val) or val is None:
        return None
    s = str(val).replace('\r', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s and s.lower() not in ('nan', 'none', 'null') else None

asset_spec_map = {str(r['id']).strip(): clean_unit_spec(r.get('spesifikasi')) for _, r in df_wrk_assets.iterrows()}
df_wrk_units['specification'] = df_wrk_units['ams_asset_id'].astype(str).map(asset_spec_map)

# Ordered consolidation targets: COMP-WORK-0001 through COMP-WORK-0002
TARGET_ORDER = [
    ('9', 'Z'),
    ('6', 'Precision Tower')
]

def map_wrk_asset(row):
    a_id = str(row.get('id', '')).strip()
    b = str(row.get('brand', '')).strip().upper()
    n = str(row.get('name', '')).strip().upper()
    
    # Asset 6335 fix (HP Z6 G4 entered under Dell)
    if a_id == '6335' or b == 'HP' or 'HP' in n or 'Z' in n:
        return ('9', 'Z')
    if b == 'DELL' or 'PRECISION' in n:
        return ('6', 'Precision Tower')
        
    raise ValueError(f"Unmapped asset: id={a_id}, brand={b}, name={n}")

df_wrk_merged = df_wrk_assets.merge(
    df_wrk_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_wrk_merged.iterrows():
    target_key = map_wrk_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val) in TARGET_ORDER:
    group_rows = groups.get((b_id, name_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': None,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r['id']).strip()] = seq_counter
        
    seq_counter += 1

# Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

# Update units lot_id and specification in staging
for idx in range(len(df_wrk_units)):
    ams_id = str(df_wrk_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_wrk_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_wrk_units.columns:
        df_wrk_units[c] = pd.to_numeric(df_wrk_units[c], errors='coerce').astype('Int64')
df_wrk_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

# Synchronize global masters & SQL scripts
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# Local Assertions
assert len(df_sub_barangs) == 2, f'Expected 2 barangs, got {len(df_sub_barangs)}'
assert len(df_wrk_units) == 46, f'Expected 46 units, got {len(df_wrk_units)}'
assert df_sub_barangs.at[0, 'name'] == 'Z', 'First barang must be Z'
assert df_sub_barangs.at[0, 'number'] == 'COMP-WORK-0001', 'First barang number must be COMP-WORK-0001'

print(f"\n[SUCCESS] Consolidated COMP-WORK from 9 barangs down to {len(df_sub_barangs)} barangs across 46 units.")



Processing Subcategory: COMP-WORK

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated COMP-WORK from 9 barangs down to 2 barangs across 46 units.


---
## 20. Subcategory ELEK-CCT: CCTV Consolidation
### Type-Based CCTV Categorization & DVR Model Normalization

**Category Rules & Approach:**
1. **Bloat Reduction**: Consolidates 36 single-camera/location records into 6 clean master barangs:
   - Package DVR (`DVR CCTV Paket`, China)
   - Standalone DVRs with brand omitted (`DVR DS-7332HUHI-K4`, `DVR DS-7208HTHI-K2`)
   - Categorized cameras (`CCTV` [Tipe Analog], `CCTV` [Tipe IP Cam], `CCTV` [Tipe Lainnya])
2. **Erroneous Record Deletion**: Asset 8200 (`SOLIS`, unit ID 2168) was an erroneous admin entry and is permanently deleted.
3. **Specification Normalization**: Locations stripped from names and specifications. Units inherit clean category specification tags.


In [21]:
# =============================================================================
# Subcategory ELEK-CCT (CCTV) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-CCT'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

# Filter out invalid Asset 8200 (SOLIS)
df_cct_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_cct_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

df_cct_assets = df_cct_assets[df_cct_assets['id'] != '8200'].copy()
df_cct_units = df_cct_units[df_cct_units['ams_asset_id'] != '8200'].copy()

# Persist cleaned assets partition and manifest
df_cct_assets.to_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', index=False)
with open(DATA_DIR / 'smart' / 'staging' / 'assets' / '_split_manifest.json', 'r', encoding='utf-8') as f:
    split_m = json.load(f)
split_m['subcategories'][SUB_CODE]['row_count'] = len(df_cct_assets)
with open(DATA_DIR / 'smart' / 'staging' / 'assets' / '_split_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(split_m, f, indent=2)

TARGET_ORDER_CCT = [
    ('52', 'DVR CCTV Paket', 'EN 6216 HV 16 port 2 TB'),
    ('75', 'DVR DS-7332HUHI-K4', '32CH 5MP HDTVI SERIES (4K & H265 Pro+)'),
    ('75', 'DVR DS-7208HTHI-K2', '8CH 8MP HDTVI SERIES (4K & H265 Pro+)'),
    ('75', 'CCTV', 'Tipe Analog'),
    ('75', 'CCTV', 'Tipe IP Cam'),
    ('75', 'CCTV', 'Tipe Lainnya')
]

def map_cct_asset(row):
    a_id = str(row.get('id', '')).strip()
    if a_id == '3640':
        return ('52', 'DVR CCTV Paket', 'EN 6216 HV 16 port 2 TB')
    if a_id == '6564':
        return ('75', 'DVR DS-7332HUHI-K4', '32CH 5MP HDTVI SERIES (4K & H265 Pro+)')
    if a_id == '6565':
        return ('75', 'DVR DS-7208HTHI-K2', '8CH 8MP HDTVI SERIES (4K & H265 Pro+)')
    
    spec = str(row.get('spesifikasi', '')).lower()
    if 'ip cam' in spec:
        return ('75', 'CCTV', 'Tipe IP Cam')
    elif 'analo' in spec or 'alano' in spec:
        return ('75', 'CCTV', 'Tipe Analog')
    else:
        return ('75', 'CCTV', 'Tipe Lainnya')

df_cct_merged = df_cct_assets.merge(
    df_cct_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_cct_merged.iterrows():
    target_key = map_cct_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_CCT:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_cct_units.index:
    ams_id = str(df_cct_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_cct_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    if ams_id in unit_spec_map:
        df_cct_units.at[idx, 'specification'] = unit_spec_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_cct_units.columns:
        df_cct_units[c] = pd.to_numeric(df_cct_units[c], errors='coerce').astype('Int64')
df_cct_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 6, f'Expected 6 barangs, got {len(df_sub_barangs)}'
assert len(df_cct_units) == 35, f'Expected 35 units, got {len(df_cct_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-CCT from 36 barangs down to {len(df_sub_barangs)} barangs across 35 units (Asset 8200 deleted).")



Processing Subcategory: ELEK-CCT

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-CCT from 36 barangs down to 6 barangs across 35 units (Asset 8200 deleted).


---
## 21. Subcategory ELEK-CS: Ceiling Speaker Consolidation
### TOA Model Normalization & Corrupted Entry Removal

**Category Rules & Approach:**
1. **Bloat Reduction**: Consolidates 6 bloated records down to 2 distinct TOA models:
   - `ZS-645R` (Daya Nominal 5 W, 78 units)
   - `ZS-658R` (Daya Nominal 6 W, 2 units)
2. **Corrupted Record Deletion**: Asset 9298 (`rusak pada tanggal 25 juli 2026`, unit ID 2273) is permanently deleted.
3. **Specification Normalization**: Wattage ratings assigned cleanly to units (`Daya Nominal 5 W` and `Daya Nominal 6 W`).


In [22]:
# =============================================================================
# Subcategory ELEK-CS (Ceiling Speaker) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-CS'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

# Filter out invalid Asset 9298 (rusak record)
df_cs_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_cs_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

df_cs_assets = df_cs_assets[df_cs_assets['id'] != '9298'].copy()
df_cs_units = df_cs_units[df_cs_units['ams_asset_id'] != '9298'].copy()

# Persist cleaned assets partition and manifest
df_cs_assets.to_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', index=False)
with open(DATA_DIR / 'smart' / 'staging' / 'assets' / '_split_manifest.json', 'r', encoding='utf-8') as f:
    split_m = json.load(f)
split_m['subcategories'][SUB_CODE]['row_count'] = len(df_cs_assets)
with open(DATA_DIR / 'smart' / 'staging' / 'assets' / '_split_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(split_m, f, indent=2)

TARGET_ORDER_CS = [
    ('55', 'ZS-645R', 'Daya Nominal 5 W'),
    ('55', 'ZS-658R', 'Daya Nominal 6 W')
]

def map_cs_asset(row):
    a_id = str(row.get('id', '')).strip()
    spec = str(row.get('spesifikasi', '')).upper()
    name = str(row.get('name', '')).upper()
    if '658' in spec or '658' in name or a_id in ('11985', '11986'):
        return ('55', 'ZS-658R', 'Daya Nominal 6 W')
    else:
        return ('55', 'ZS-645R', 'Daya Nominal 5 W')

df_cs_merged = df_cs_assets.merge(
    df_cs_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_cs_merged.iterrows():
    target_key = map_cs_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_CS:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_cs_units.index:
    ams_id = str(df_cs_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_cs_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    if ams_id in unit_spec_map:
        df_cs_units.at[idx, 'specification'] = unit_spec_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_cs_units.columns:
        df_cs_units[c] = pd.to_numeric(df_cs_units[c], errors='coerce').astype('Int64')
df_cs_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 2, f'Expected 2 barangs, got {len(df_sub_barangs)}'
assert len(df_cs_units) == 80, f'Expected 80 units, got {len(df_cs_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-CS from 6 barangs down to {len(df_sub_barangs)} barangs across 80 units (Asset 9298 deleted).")



Processing Subcategory: ELEK-CS

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-CS from 6 barangs down to 2 barangs across 80 units (Asset 9298 deleted).


---
## 22. Subcategory ELEK-PCT: Paging Control TOA Consolidation
### Code Renaming (ELEK-FCT ➔ ELEK-PCT) & Model Simplification

**Category Rules & Approach:**
1. **Code & Name Correction**: Code updated to `ELEK-PCT` and subcategory name to `Paging Control TOA`.
2. **Name Simplification**: Barang name stripped of redundant subcategory and brand prefixes, simplified to `RM-200M`.
3. **Specification**: Preserved specification `Zone Control` on barang and unit.


In [23]:
# =============================================================================
# Subcategory ELEK-PCT (Paging Control TOA) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-PCT'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_pct_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_pct_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_PCT = [
    ('55', 'RM-200M', 'Zone Control')
]

df_pct_merged = df_pct_assets.merge(
    df_pct_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_PCT:
    group_rows = [r for _, r in df_pct_merged.iterrows()]
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_pct_units.index:
    ams_id = str(df_pct_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_pct_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    if ams_id in unit_spec_map:
        df_pct_units.at[idx, 'specification'] = unit_spec_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_pct_units.columns:
        df_pct_units[c] = pd.to_numeric(df_pct_units[c], errors='coerce').astype('Int64')
df_pct_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 1, f'Expected 1 barang, got {len(df_sub_barangs)}'
assert len(df_pct_units) == 1, f'Expected 1 unit, got {len(df_pct_units)}'
assert df_sub_barangs.at[0, 'name'] == 'RM-200M'
print(f"\n[SUCCESS] Consolidated ELEK-PCT to {len(df_sub_barangs)} barang across {len(df_pct_units)} unit.")



Processing Subcategory: ELEK-PCT

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-PCT to 1 barang across 1 unit.


---
## 23. Subcategory ELEK-FR: Fingerprint Reader Consolidation
### Brand Distinction & Hardware Normalization

**Category Rules & Approach:**
1. **Bloat Reduction**: Consolidates 3 single-unit records into 2 clean master barangs:
   - `Fingkey Access` (Nitgen, 3 units: merged lots 0001 and 0003)
   - `Fingerprint (Lawas)` (Solution, 1 unit: renamed lot 0002)
2. **Specification**: Kept as None (no custom hardware spec needed).


In [24]:
# =============================================================================
# Subcategory ELEK-FR (Fingerprint Reader) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-FR'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_fr_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_fr_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_FR = [
    ('124', 'Fingkey Access', None),
    ('133', 'Fingerprint (Lawas)', None)
]

def map_fr_asset(row):
    brand = str(row.get('brand', '')).upper()
    a_id = str(row.get('id', '')).strip()
    if a_id == '8948' or 'SOLUTION' in brand:
        return ('133', 'Fingerprint (Lawas)', None)
    else:
        return ('124', 'Fingkey Access', None)

df_fr_merged = df_fr_assets.merge(
    df_fr_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_fr_merged.iterrows():
    target_key = map_fr_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_FR:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_fr_units.index:
    ams_id = str(df_fr_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_fr_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_fr_units.at[idx, 'specification'] = None

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_fr_units.columns:
        df_fr_units[c] = pd.to_numeric(df_fr_units[c], errors='coerce').astype('Int64')
df_fr_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 2, f'Expected 2 barangs, got {len(df_sub_barangs)}'
assert len(df_fr_units) == 4, f'Expected 4 units, got {len(df_fr_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-FR from 3 barangs down to {len(df_sub_barangs)} barangs across 4 units.")



Processing Subcategory: ELEK-FR

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-FR from 3 barangs down to 2 barangs across 4 units.


---
## 24. Subcategory ELEK-PVC: Paging Volume Control Consolidation
### Code Renaming (ELEK-FVC ➔ ELEK-PVC) & Name Simplification

**Category Rules & Approach:**
1. **Code & Name Correction**: Code updated to `ELEK-PVC` and subcategory name to `Paging Volume Control`.
2. **Name Simplification**: Barang name simplified to product code `DS`.
3. **Specification**: Preserved specification `Volume Control lantai 1,2,3,4` on barang and unit.


In [25]:
# =============================================================================
# Subcategory ELEK-PVC (Paging Volume Control) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-PVC'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_pvc_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_pvc_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_PVC = [
    ('75', 'DS', 'Volume Control lantai 1,2,3,4')
]

df_pvc_merged = df_pvc_assets.merge(
    df_pvc_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_PVC:
    group_rows = [r for _, r in df_pvc_merged.iterrows()]
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_pvc_units.index:
    ams_id = str(df_pvc_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_pvc_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    if ams_id in unit_spec_map:
        df_pvc_units.at[idx, 'specification'] = unit_spec_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_pvc_units.columns:
        df_pvc_units[c] = pd.to_numeric(df_pvc_units[c], errors='coerce').astype('Int64')
df_pvc_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 1, f'Expected 1 barang, got {len(df_sub_barangs)}'
assert len(df_pvc_units) == 1, f'Expected 1 unit, got {len(df_pvc_units)}'
assert df_sub_barangs.at[0, 'name'] == 'DS'
print(f"\n[SUCCESS] Consolidated ELEK-PVC to {len(df_sub_barangs)} barang across {len(df_pvc_units)} unit.")



Processing Subcategory: ELEK-PVC

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-PVC to 1 barang across 1 unit.


---
## 25. Subcategory ELEK-HD: Heat Detector Consolidation
### Hong Chang Model Deduplication & Location Removal

**Category Rules & Approach:**
1. **Bloat Reduction**: Massive bloat reduction from 113 location-based records down to 2 distinct Hong Chang models:
   - `HC-306A` (Voltage : DC 12 V-30V (DC 24v) Alarm Current 60 mA, 156 units: consolidated 148 explicit + 8 chronological null-spec units)
   - `HC-202` (Asset 11873, 1 unit)
2. **Location Removal**: All corridor and room location labels removed from master names and specifications.
3. **Unit Specification**: Units preserve voltage rating specification if present.


In [26]:
# =============================================================================
# Subcategory ELEK-HD (Heat Detector) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-HD'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_hd_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_hd_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_HD = [
    ('107', 'HC-306A', 'Voltage : DC 12 V-30V (DC 24v) Alarm Current 60 mA'),
    ('107', 'HC-202', None)
]

def map_hd_asset(row):
    a_id = str(row.get('id', '')).strip()
    spec = str(row.get('spesifikasi', ''))
    name = str(row.get('name', ''))
    if a_id == '11873' or '202' in spec or '202' in name:
        return ('107', 'HC-202', None)
    else:
        return ('107', 'HC-306A', 'Voltage : DC 12 V-30V (DC 24v) Alarm Current 60 mA')

df_hd_merged = df_hd_assets.merge(
    df_hd_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_hd_merged.iterrows():
    target_key = map_hd_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_HD:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        raw_s = str(r.get('spesifikasi', ''))
        if 'voltage' in raw_s.lower() or 'dc 12' in raw_s.lower():
            unit_spec_map[aid] = 'Voltage : DC 12 V-30V (DC 24v) Alarm Current 60 mA'
        else:
            unit_spec_map[aid] = None
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_hd_units.index:
    ams_id = str(df_hd_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_hd_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_hd_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_hd_units.columns:
        df_hd_units[c] = pd.to_numeric(df_hd_units[c], errors='coerce').astype('Int64')
df_hd_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 2, f'Expected 2 barangs, got {len(df_sub_barangs)}'
assert len(df_hd_units) == 157, f'Expected 157 units, got {len(df_hd_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-HD from 113 barangs down to {len(df_sub_barangs)} barangs across 157 units.")



Processing Subcategory: ELEK-HD

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-HD from 113 barangs down to 2 barangs across 157 units.


---
## 26. ELEK Range Consolidation Verification
### Integrity Checks for ELEK-ATS through ELEK-HD

Runs validation across master tables and newly consolidated subcategories.


In [27]:
# =============================================================================
# Final Quality Assertions: ELEK Category Range (ELEK-ATS to ELEK-HD)
# =============================================================================
print(f"\n{'='*80}")
print("Verifying ELEK Category Range Masters & Consistency")
print(f"{'='*80}\n")

# 1. Subcategories check
df_sub_chk = pd.read_csv(FINAL_SUBCATEGORY_PATH)
assert df_sub_chk.loc[df_sub_chk['id'] == 93, 'code'].values[0] == 'ELEK-PCT'
assert df_sub_chk.loc[df_sub_chk['id'] == 93, 'name'].values[0] == 'Paging Control TOA'
assert df_sub_chk.loc[df_sub_chk['id'] == 92, 'code'].values[0] == 'ELEK-PVC'
assert df_sub_chk.loc[df_sub_chk['id'] == 92, 'name'].values[0] == 'Paging Volume Control'

# 2. Partition barangs & units check
assert len(pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-CCT.csv')) == 6
assert len(pd.read_csv(STAGING_UNITS_DIR / 'ELEK-CCT.csv')) == 35
assert 8200 not in pd.read_csv(STAGING_UNITS_DIR / 'ELEK-CCT.csv')['ams_asset_id'].astype(int).values

assert len(pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-CS.csv')) == 2
assert len(pd.read_csv(STAGING_UNITS_DIR / 'ELEK-CS.csv')) == 80
assert 9298 not in pd.read_csv(STAGING_UNITS_DIR / 'ELEK-CS.csv')['ams_asset_id'].astype(int).values

assert len(pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-PCT.csv')) == 1
assert len(pd.read_csv(STAGING_UNITS_DIR / 'ELEK-PCT.csv')) == 1
assert pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-PCT.csv').at[0, 'name'] == 'RM-200M'

assert len(pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-FR.csv')) == 2
assert len(pd.read_csv(STAGING_UNITS_DIR / 'ELEK-FR.csv')) == 4

assert len(pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-PVC.csv')) == 1
assert len(pd.read_csv(STAGING_UNITS_DIR / 'ELEK-PVC.csv')) == 1
assert pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-PVC.csv').at[0, 'name'] == 'DS'

assert len(pd.read_csv(STAGING_BARANGS_DIR / 'ELEK-HD.csv')) == 2
assert len(pd.read_csv(STAGING_UNITS_DIR / 'ELEK-HD.csv')) == 157

# 3. Global Master Tables
df_barangs_m = pd.read_csv(FINAL_BARANG_DIR / 'barangs.csv')
df_lots_m = pd.read_csv(FINAL_LOT_DIR / 'lots.csv')
df_units_m = pd.read_csv(FINAL_UNIT_DIR / 'units.csv')

assert len(df_units_m) == 6941, f'Expected 6,941 units, got {len(df_units_m)}'
assert len(df_barangs_m) == len(df_lots_m)
assert (df_barangs_m['id'].astype(int) == list(range(1, len(df_barangs_m) + 1))).all()
assert (df_lots_m['id'].astype(int) == list(range(1, len(df_lots_m) + 1))).all()

print(f"[SUCCESS] All ELEK range assertions passed! Master barangs: {len(df_barangs_m):,}, units: {len(df_units_m):,}.")



Verifying ELEK Category Range Masters & Consistency

[SUCCESS] All ELEK range assertions passed! Master barangs: 888, units: 6,941.


---
## 27. Subcategory ELEK-KUL: Kulkas Normalization
### Brand Distinction & Title Case Normalization

**Category Rules & Approach:**
1. **Brand-Based Separation**: Retains 1 barang per brand across Toshiba (7), Sharp (88), and Nasional (141).
2. **Title Case Standardization**: Standardizes names to `Kulkas` and specifications to `None`.

In [28]:
# =============================================================================
# Subcategory ELEK-KUL (Kulkas) Normalization
# =============================================================================
SUB_CODE = 'ELEK-KUL'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_kul_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_kul_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_KUL = [
    ('7', 'Kulkas', None),
    ('88', 'Kulkas', None),
    ('141', 'Kulkas', None)
]

def map_kul_asset(row):
    b = str(row.get('brand', '')).upper()
    if 'TOSHIBA' in b:
        return ('7', 'Kulkas', None)
    elif 'SHARP' in b:
        return ('88', 'Kulkas', None)
    elif 'NASIONAL' in b:
        return ('141', 'Kulkas', None)
    raise ValueError(f"Unknown brand for KUL asset: {row}")

df_kul_merged = df_kul_assets.merge(
    df_kul_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_kul_merged.iterrows():
    target_key = map_kul_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_KUL:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_kul_units.index:
    ams_id = str(df_kul_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_kul_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_kul_units.columns:
        df_kul_units[c] = pd.to_numeric(df_kul_units[c], errors='coerce').astype('Int64')
df_kul_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 3, f'Expected 3 barangs, got {len(df_sub_barangs)}'
assert len(df_kul_units) == 8, f'Expected 8 units, got {len(df_kul_units)}'
print(f"\n[SUCCESS] Normalized ELEK-KUL to {len(df_sub_barangs)} barangs across {len(df_kul_units)} units.")


Processing Subcategory: ELEK-KUL

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Normalized ELEK-KUL to 3 barangs across 8 units.


---
## 28. Subcategory ELEK-MD: Multigas Detector Consolidation
### Consolidation into Single BW Microchip XL Master Item

**Category Rules & Approach:**
1. **Model Consolidation**: Merges 2 redundant barang entries into a single master record `BW Microchip XL` (Honeywell [140]).
2. **Technical Specification**: Captures sensor configuration `O2/LEL/H2S/CO, OE Version` at master and unit levels.

In [29]:
# =============================================================================
# Subcategory ELEK-MD (Multigas Detector) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-MD'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_md_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_md_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_MD = [
    ('140', 'BW Microchip XL', 'O2/LEL/H2S/CO, OE Version')
]

def map_md_asset(row):
    return ('140', 'BW Microchip XL', 'O2/LEL/H2S/CO, OE Version')

df_md_merged = df_md_assets.merge(
    df_md_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_md_merged.iterrows():
    target_key = map_md_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_MD:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_md_units.index:
    ams_id = str(df_md_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_md_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_md_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_md_units.columns:
        df_md_units[c] = pd.to_numeric(df_md_units[c], errors='coerce').astype('Int64')
df_md_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 1, f'Expected 1 barang, got {len(df_sub_barangs)}'
assert len(df_md_units) == 3, f'Expected 3 units, got {len(df_md_units)}'
assert df_sub_barangs.at[0, 'name'] == 'BW Microchip XL'
print(f"\n[SUCCESS] Consolidated ELEK-MD from 2 barangs down to 1 barang across 3 units.")


Processing Subcategory: ELEK-MD

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-MD from 2 barangs down to 1 barang across 3 units.


---
## 29. Subcategory ELEK-MIC: Microwave Consolidation
### Model & Capacity Normalization

**Category Rules & Approach:**
1. **Krisbow Normalization**: Renames 0001 (Krisbow [64]) to `Kris` with specification `20 Ltr`.
2. **Sharp Normalization**: Renames 0002 (Sharp [88]) to `R-728` with specification `None`.

In [30]:
# =============================================================================
# Subcategory ELEK-MIC (Microwave) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-MIC'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_mic_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_mic_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_MIC = [
    ('64', 'Kris', '20 Ltr'),
    ('88', 'R-728', None)
]

def map_mic_asset(row):
    b = str(row.get('brand', '')).upper()
    if 'KRISBOW' in b:
        return ('64', 'Kris', '20 Ltr')
    elif 'SHARP' in b:
        return ('88', 'R-728', None)
    raise ValueError(f"Unknown brand for MIC asset: {row}")

df_mic_merged = df_mic_assets.merge(
    df_mic_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_mic_merged.iterrows():
    target_key = map_mic_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_MIC:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_mic_units.index:
    ams_id = str(df_mic_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_mic_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_mic_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_mic_units.columns:
        df_mic_units[c] = pd.to_numeric(df_mic_units[c], errors='coerce').astype('Int64')
df_mic_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 2, f'Expected 2 barangs, got {len(df_sub_barangs)}'
assert len(df_mic_units) == 7, f'Expected 7 units, got {len(df_mic_units)}'
print(f"\n[SUCCESS] Normalized ELEK-MIC to {len(df_sub_barangs)} barangs across {len(df_mic_units)} units.")


Processing Subcategory: ELEK-MIC

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Normalized ELEK-MIC to 2 barangs across 7 units.


---
## 30. Subcategory ELEK-MK: Mesin Kopi Normalization
### Title Case Normalization

**Category Rules & Approach:**
1. **Title Case Standardization**: Standardizes name to `Mesin Kopi` and specification to `Mesin Kopi Untuk Cafe`.

In [31]:
# =============================================================================
# Subcategory ELEK-MK (Mesin Kopi) Normalization
# =============================================================================
SUB_CODE = 'ELEK-MK'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_mk_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_mk_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_MK = [
    ('104', 'Mesin Kopi', 'Mesin Kopi Untuk Cafe')
]

def map_mk_asset(row):
    return ('104', 'Mesin Kopi', 'Mesin Kopi Untuk Cafe')

df_mk_merged = df_mk_assets.merge(
    df_mk_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_mk_merged.iterrows():
    target_key = map_mk_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_MK:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_mk_units.index:
    ams_id = str(df_mk_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_mk_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_mk_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_mk_units.columns:
        df_mk_units[c] = pd.to_numeric(df_mk_units[c], errors='coerce').astype('Int64')
df_mk_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 1, f'Expected 1 barang, got {len(df_sub_barangs)}'
assert len(df_mk_units) == 1, f'Expected 1 unit, got {len(df_mk_units)}'
print(f"\n[SUCCESS] Normalized ELEK-MK to {len(df_sub_barangs)} barang across {len(df_mk_units)} unit.")


Processing Subcategory: ELEK-MK

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Normalized ELEK-MK to 1 barang across 1 unit.


---
## 31. Subcategory ELEK-MPK: Mesin Penggiling Kopi Normalization
### Model Renaming & Title Case Normalization

**Category Rules & Approach:**
1. **Model Name Simplification**: Renames barang to `FGM-800AC` (Ferrarti Ferro [104]).
2. **Title Case Standardization**: Formats specification to `Mesin Blender Untuk Biji Kopi Warna Hijau`.

In [32]:
# =============================================================================
# Subcategory ELEK-MPK (Mesin Penggiling Kopi) Normalization
# =============================================================================
SUB_CODE = 'ELEK-MPK'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_mpk_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_mpk_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_MPK = [
    ('104', 'FGM-800AC', 'Mesin Blender Untuk Biji Kopi Warna Hijau')
]

def map_mpk_asset(row):
    return ('104', 'FGM-800AC', 'Mesin Blender Untuk Biji Kopi Warna Hijau')

df_mpk_merged = df_mpk_assets.merge(
    df_mpk_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_mpk_merged.iterrows():
    target_key = map_mpk_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_MPK:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_mpk_units.index:
    ams_id = str(df_mpk_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_mpk_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_mpk_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_mpk_units.columns:
        df_mpk_units[c] = pd.to_numeric(df_mpk_units[c], errors='coerce').astype('Int64')
df_mpk_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 1, f'Expected 1 barang, got {len(df_sub_barangs)}'
assert len(df_mpk_units) == 1, f'Expected 1 unit, got {len(df_mpk_units)}'
print(f"\n[SUCCESS] Normalized ELEK-MPK to {len(df_sub_barangs)} barang across {len(df_mpk_units)} unit.")


Processing Subcategory: ELEK-MPK

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Normalized ELEK-MPK to 1 barang across 1 unit.


---
## 32. Subcategory ELEK-SD: Smoke Detector Consolidation
### Hong Chang Model Deduplication & Location Removal

**Category Rules & Approach:**
1. **Deduplication**: Massive reduction from 54 location-based entries down to 5 distinct Hong Chang [107] models:
   - `HC-206A` (46 units): Standardized electrical specification `Supply Voltage : DC 24V Standby Current : 60 A Alarm Current 60 mA`.
   - `HC-206E/E-R` (21 units): Combined base and -R variant models.
   - `HC-202D` (3 units).
   - `FST-718` (2 units).
   - `Smoke Detector (Tipe Lainnya)` (5 units): Fallback for unspecified assets.
2. **Location Removal**: All room, corridor, and ceiling tags stripped from master barang names and specifications.

In [33]:
# =============================================================================
# Subcategory ELEK-SD (Smoke Detector) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-SD'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_sd_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_sd_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_SD = [
    ('107', 'HC-206A', 'Supply Voltage : DC 24V Standby Current : 60 A Alarm Current 60 mA'),
    ('107', 'HC-206E/E-R', None),
    ('107', 'HC-202D', None),
    ('107', 'FST-718', None),
    ('107', 'Smoke Detector (Tipe Lainnya)', None)
]

def map_sd_asset(row):
    a_id = str(row.get('id', '')).strip()
    spec = str(row.get('spesifikasi', '') or '')
    name = str(row.get('name', '') or '')
    spec_l = spec.lower()
    name_l = name.lower()
    
    if a_id in ['8435', '11870', '11874'] or 'hc202' in spec_l or 'hc-202' in spec_l or 'hc 202' in spec_l:
        return ('107', 'HC-202D', None)
    if a_id in ['11871', '11872'] or '718' in spec_l:
        return ('107', 'FST-718', None)
    if (a_id in ['11885', '11972', '12003', '12005', '12007', '12011', '12012', '12013', '12043'] 
        or '206e' in spec_l or '206e' in name_l 
        or (int(a_id) in range(11891, 11906) and '206e' in spec_l)):
        return ('107', 'HC-206E/E-R', None)
    if a_id in ['9452', '10550', '10652', '11881', '11888']:
        return ('107', 'Smoke Detector (Tipe Lainnya)', None)
    return ('107', 'HC-206A', 'Supply Voltage : DC 24V Standby Current : 60 A Alarm Current 60 mA')

df_sd_merged = df_sd_assets.merge(
    df_sd_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_sd_merged.iterrows():
    target_key = map_sd_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_SD:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_sd_units.index:
    ams_id = str(df_sd_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_sd_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_sd_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_sd_units.columns:
        df_sd_units[c] = pd.to_numeric(df_sd_units[c], errors='coerce').astype('Int64')
df_sd_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 5, f'Expected 5 barangs, got {len(df_sub_barangs)}'
assert len(df_sd_units) == 77, f'Expected 77 units, got {len(df_sd_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-SD from 54 barangs down to {len(df_sub_barangs)} barangs across {len(df_sd_units)} units.")


Processing Subcategory: ELEK-SD

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-SD from 54 barangs down to 5 barangs across 77 units.


---
## 33. Subcategory ELEK-SPE: Speaker Consolidation
### Model Renaming, JBL Consolidation & Location Cleanup

**Category Rules & Approach:**
1. **Model Renaming**:
   - 0001 (Altec Lansing [18]) ➔ `Speaker 2:1`, spec: `None`.
   - 0002 (Krezt [56]) ➔ `WAS-010 UHF`, spec: `None`.
   - 0003 (Samsung [29]) ➔ `Paging 1`, location stripped from spec.
   - 0005 (Maximum [118]) ➔ `MAX 112 230 EB`, spec: `None`.
   - 0006 (Roland [116]) ➔ `KC-200`, spec: `AC 230 V - 50/60 Hz`.
   - 0007 (Boss [117]) ➔ `93120.2`, spec: `HWPW-VC/CC PB, MDF, T-MDF`.
   - 0008 (Samson [115]) ➔ `Auro D415`, spec: `400 Watt Powered Speaker Cabinet`.
   - 0012 (Laney [145]) & 0013 (Behringer [146]) ➔ `Speaker Aktif`.
   - 0015 (TOA [55]) ➔ `ZR`.
2. **JBL Models Consolidation**:
   - Combine 0004 & 0009 ➔ `Standing` (2 units).
   - Combine 0010, 0011 & 0014 ➔ `Bluetooth` (5 units).

In [34]:
# =============================================================================
# Subcategory ELEK-SPE (Speaker) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-SPE'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_spe_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_spe_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_SPE = [
    ('18', 'Speaker 2:1', None),
    ('56', 'WAS-010 UHF', None),
    ('29', 'Paging 1', None),
    ('114', 'Standing', None),
    ('118', 'MAX 112 230 EB', None),
    ('116', 'KC-200', 'AC 230 V - 50/60 Hz'),
    ('117', '93120.2', 'HWPW-VC/CC PB, MDF, T-MDF'),
    ('115', 'Auro D415', '400 Watt Powered Speaker Cabinet'),
    ('114', 'Bluetooth', None),
    ('145', 'Speaker Aktif', None),
    ('146', 'Speaker Aktif', None),
    ('55', 'ZR', None)
]

def map_spe_asset(row):
    aid = str(row.get('id', '')).strip()
    if aid == '2983':
        return ('18', 'Speaker 2:1', None)
    if aid == '3757':
        return ('56', 'WAS-010 UHF', None)
    if aid == '8141':
        return ('29', 'Paging 1', None)
    if aid in ['8598', '10500']:
        return ('114', 'Standing', None)
    if aid == '8599':
        return ('118', 'MAX 112 230 EB', None)
    if aid == '8600':
        return ('116', 'KC-200', 'AC 230 V - 50/60 Hz')
    if aid == '8601':
        return ('117', '93120.2', 'HWPW-VC/CC PB, MDF, T-MDF')
    if aid == '8602':
        return ('115', 'Auro D415', '400 Watt Powered Speaker Cabinet')
    if aid in ['10501', '10502', '10503', '10504', '10558']:
        return ('114', 'Bluetooth', None)
    if aid == '10505':
        return ('145', 'Speaker Aktif', None)
    if aid == '10506':
        return ('146', 'Speaker Aktif', None)
    if aid == '11808':
        return ('55', 'ZR', None)
    raise ValueError(f"Unhandled SPE asset: {row}")

df_spe_merged = df_spe_assets.merge(
    df_spe_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_spe_merged.iterrows():
    target_key = map_spe_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_SPE:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_spe_units.index:
    ams_id = str(df_spe_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_spe_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_spe_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_spe_units.columns:
        df_spe_units[c] = pd.to_numeric(df_spe_units[c], errors='coerce').astype('Int64')
df_spe_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 12, f'Expected 12 barangs, got {len(df_sub_barangs)}'
assert len(df_spe_units) == 17, f'Expected 17 units, got {len(df_spe_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-SPE from 15 barangs down to {len(df_sub_barangs)} barangs across {len(df_spe_units)} units.")


Processing Subcategory: ELEK-SPE

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-SPE from 15 barangs down to 12 barangs across 17 units.


---
## 34. Subcategory ELEK-SPR: Sprinkler Consolidation
### Consolidation into Dry Pipe & Sprinkler (Tipe Lainnya)

**Category Rules & Approach:**
1. **Dry Pipe Consolidation**: Combines 0001 (`Tipe Dry Pipe`) and 0002 (`Tipe dri Pipe`) into a single master barang `Dry Pipe` (8 units).
2. **Fallback Categorization**: Renames 0003 to `Sprinkler (Tipe Lainnya)` (143 units).

In [35]:
# =============================================================================
# Subcategory ELEK-SPR (Sprinkler) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-SPR'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_spr_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_spr_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_SPR = [
    ('107', 'Dry Pipe', None),
    ('107', 'Sprinkler (Tipe Lainnya)', None)
]

def map_spr_asset(row):
    spec = str(row.get('spesifikasi', '') or '')
    if 'dry pipe' in spec.lower() or 'dri pipe' in spec.lower():
        return ('107', 'Dry Pipe', None)
    return ('107', 'Sprinkler (Tipe Lainnya)', None)

df_spr_merged = df_spr_assets.merge(
    df_spr_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_spr_merged.iterrows():
    target_key = map_spr_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_SPR:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_spr_units.index:
    ams_id = str(df_spr_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_spr_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_spr_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_spr_units.columns:
        df_spr_units[c] = pd.to_numeric(df_spr_units[c], errors='coerce').astype('Int64')
df_spr_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 2, f'Expected 2 barangs, got {len(df_sub_barangs)}'
assert len(df_spr_units) == 151, f'Expected 151 units, got {len(df_spr_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-SPR from 3 barangs down to {len(df_sub_barangs)} barangs across {len(df_spr_units)} units.")


Processing Subcategory: ELEK-SPR

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-SPR from 3 barangs down to 2 barangs across 151 units.


---
## 35. Subcategory ELEK-TG: Total Station & Geodesi Consolidation
### Aluminium Tripod Merger & Title Case Normalization

**Category Rules & Approach:**
1. **Tripod Consolidation**: Merges 0002 and 0003 into `Aluminium Tripod` (Topcon [95], 2 units), removing transient project descriptions from specifications.
2. **Title Case Standardization**: Standardizes equipment names to clean Title Case.

In [36]:
# =============================================================================
# Subcategory ELEK-TG (Total Station & Geodesi) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-TG'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_tg_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_tg_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_TG = [
    ('95', 'Total Station Topcon GM 52', None),
    ('95', 'Aluminium Tripod', None),
    ('95', 'Prisma Polygon', None),
    ('95', 'Auto Level Topcon ATB4A', None),
    ('95', 'Baterai Total Station Topcon GM 51', None),
    ('95', 'USB', None),
    ('95', 'Tali Tas Alat', None),
    ('95', 'Software', None),
    ('95', 'Aksesori Reflektor', None),
    ('95', 'Topografi', None)
]

def map_tg_asset(row):
    aid = str(row.get('id', '')).strip()
    if aid == '8124':
        return ('95', 'Total Station Topcon GM 52', None)
    if aid in ['8125', '8126']:
        return ('95', 'Aluminium Tripod', None)
    if aid == '8127':
        return ('95', 'Prisma Polygon', None)
    if aid == '8128':
        return ('95', 'Auto Level Topcon ATB4A', None)
    if aid == '8129':
        return ('95', 'Baterai Total Station Topcon GM 51', None)
    if aid == '8130':
        return ('95', 'USB', None)
    if aid == '8131':
        return ('95', 'Tali Tas Alat', None)
    if aid == '8132':
        return ('95', 'Software', None)
    if aid == '8133':
        return ('95', 'Aksesori Reflektor', None)
    if aid == '11916':
        return ('95', 'Topografi', None)
    raise ValueError(f"Unhandled TG asset: {row}")

df_tg_merged = df_tg_assets.merge(
    df_tg_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_tg_merged.iterrows():
    target_key = map_tg_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_TG:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_tg_units.index:
    ams_id = str(df_tg_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_tg_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_tg_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_tg_units.columns:
        df_tg_units[c] = pd.to_numeric(df_tg_units[c], errors='coerce').astype('Int64')
df_tg_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 10, f'Expected 10 barangs, got {len(df_sub_barangs)}'
assert len(df_tg_units) == 11, f'Expected 11 units, got {len(df_tg_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-TG from 11 barangs down to {len(df_sub_barangs)} barangs across {len(df_tg_units)} units.")


Processing Subcategory: ELEK-TG

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-TG from 11 barangs down to 10 barangs across 11 units.


---
## 36. Subcategory ELEK-TV: Televisi Consolidation
### Model Simplification & TV (Size) Format Standardization

**Category Rules & Approach:**
1. **Model Name Simplification**: Strips brand and redundant text from names (e.g. `55UM7300PTA` for LG [30], `50Q80C` for Samsung [29]).
2. **Consolidation of Duplicates**: Merges multiple entries sharing the same name per brand (e.g. Samsung `TV (Kecil)` across 7 units).
3. **Format Standardization**: Standardizes format to `TV (size)` (e.g. `TV (Kecil)`, `TV (Besar)`, `TV (32 Inch)`).

In [37]:
# =============================================================================
# Subcategory ELEK-TV (Televisi) Consolidation
# =============================================================================
SUB_CODE = 'ELEK-TV'

print(f"\n{'='*80}")
print(f"Processing Subcategory: {SUB_CODE}")
print(f"{'='*80}\n")

df_tv_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_tv_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

TARGET_ORDER_TV = [
    ('8', 'Plasma TV', 'Plasma Monitor 49"'),
    ('9', 'TV (Kecil)', None),
    ('25', 'TV (Besar)', None),
    ('25', 'TV', None),
    ('25', 'TV (Kecil)', None),
    ('30', '55UM7300PTA', 'LED 55 Inch UHD 4K LED TV Smart TV ThinQ AI 55UM7300'),
    ('30', 'TV (Kecil)', None),
    ('30', 'TV (Besar)', None),
    ('161', 'TV (Besar)', None),
    ('161', 'TV (32 Inch)', None),
    ('29', '50Q80C', 'QLED UHD 4K Smart TV 50 Inch'),
    ('29', 'TV (Besar)', None),
    ('29', 'TV (Kecil)', None),
    ('29', 'TV (32 Inch)', None)
]

def map_tv_asset(row):
    aid = str(row.get('id', '')).strip()
    if aid == '2661':
        return ('8', 'Plasma TV', 'Plasma Monitor 49"')
    if aid == '9369':
        return ('9', 'TV (Kecil)', None)
    if aid == '9289':
        return ('25', 'TV (Besar)', None)
    if aid == '9394':
        return ('25', 'TV', None)
    if aid == '11964':
        return ('25', 'TV (Kecil)', None)
    if aid in ['6563', '7941']:
        return ('30', '55UM7300PTA', 'LED 55 Inch UHD 4K LED TV Smart TV ThinQ AI 55UM7300')
    if aid in ['7954', '9374', '11836']:
        return ('30', 'TV (Kecil)', None)
    if aid in ['9425', '11970']:
        return ('30', 'TV (Besar)', None)
    if aid in ['11786', '11787']:
        return ('161', 'TV (Besar)', None)
    if aid == '12036':
        return ('161', 'TV (32 Inch)', None)
    if aid == '8118':
        return ('29', '50Q80C', 'QLED UHD 4K Smart TV 50 Inch')
    if aid in ['11835', '11883', '11966']:
        return ('29', 'TV (Besar)', None)
    if aid in ['8561', '8562', '8844', '9096', '10685', '11834', '11877']:
        return ('29', 'TV (Kecil)', None)
    if aid == '8573':
        return ('29', 'TV (32 Inch)', None)
    raise ValueError(f"Unhandled TV asset: {row}")

df_tv_merged = df_tv_assets.merge(
    df_tv_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

groups = defaultdict(list)
for _, r in df_tv_merged.iterrows():
    target_key = map_tv_asset(r)
    groups[target_key].append(r)

sub_barangs = []
sub_lots = []
unit_lot_map = {}
unit_spec_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_val) in TARGET_ORDER_TV:
    group_rows = groups.get((b_id, name_val, spec_val), [])
    if not group_rows:
        continue
        
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': name_val,
        'specification': spec_val,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        aid = str(r['id']).strip()
        unit_lot_map[aid] = seq_counter
        unit_spec_map[aid] = spec_val
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

for idx in df_tv_units.index:
    ams_id = str(df_tv_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_tv_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]
    df_tv_units.at[idx, 'specification'] = unit_spec_map.get(ams_id)

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_tv_units.columns:
        df_tv_units[c] = pd.to_numeric(df_tv_units[c], errors='coerce').astype('Int64')
df_tv_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 14, f'Expected 14 barangs, got {len(df_sub_barangs)}'
assert len(df_tv_units) == 27, f'Expected 27 units, got {len(df_tv_units)}'
print(f"\n[SUCCESS] Consolidated ELEK-TV from 16 barangs down to {len(df_sub_barangs)} barangs across {len(df_tv_units)} units.")


Processing Subcategory: ELEK-TV

 Master Sync Complete: 888 barangs, 888 lots, 6,941 units.

[SUCCESS] Consolidated ELEK-TV from 16 barangs down to 14 barangs across 27 units.


---
## 37. Complete ELEK Category Consolidation Verification
### Final Integrity Assertions Across All 19 Subcategories

Runs global validation across all partition files, master tables, and referential integrity.

In [38]:
# =============================================================================
# Final Quality Assertions: Full ELEK Category Range (ELEK-ATS to ELEK-TV)
# =============================================================================
print(f"\n{'='*80}")
print("Verifying Complete ELEK Category Masters & Consistency")
print(f"{'='*80}\n")

expected_units_elek = {
    'ELEK-ATS': 1,
    'ELEK-CCT': 35,
    'ELEK-CP': 1,
    'ELEK-CS': 80,
    'ELEK-EXH': 1,
    'ELEK-FR': 4,
    'ELEK-HD': 157,
    'ELEK-KUL': 8,
    'ELEK-MD': 3,
    'ELEK-MIC': 7,
    'ELEK-MK': 1,
    'ELEK-MPK': 1,
    'ELEK-PCT': 1,
    'ELEK-PVC': 1,
    'ELEK-SD': 77,
    'ELEK-SPE': 17,
    'ELEK-SPR': 151,
    'ELEK-TG': 11,
    'ELEK-TV': 27,
}

for sub_c, exp_count in expected_units_elek.items():
    df_u_part = pd.read_csv(STAGING_UNITS_DIR / f'{sub_c}.csv')
    df_b_part = pd.read_csv(STAGING_BARANGS_DIR / f'{sub_c}.csv')
    df_l_part = pd.read_csv(STAGING_LOTS_DIR / f'{sub_c}.csv')
    assert len(df_u_part) == exp_count, f"Mismatch in {sub_c}: expected {exp_count} units, got {len(df_u_part)}"
    assert len(df_b_part) == len(df_l_part), f"Barangs vs Lots mismatch in {sub_c}: {len(df_b_part)} vs {len(df_l_part)}"
    assert df_u_part['lot_id'].isin(df_l_part['id']).all(), f"Invalid lot_id found in {sub_c} units!"

# Global Master Tables
df_barangs_m = pd.read_csv(FINAL_BARANG_DIR / 'barangs.csv')
df_lots_m = pd.read_csv(FINAL_LOT_DIR / 'lots.csv')
df_units_m = pd.read_csv(FINAL_UNIT_DIR / 'units.csv')

assert len(df_units_m) == 6941, f'Expected 6,941 units, got {len(df_units_m)}'
assert len(df_barangs_m) == len(df_lots_m)
assert (df_barangs_m['id'].astype(int) == list(range(1, len(df_barangs_m) + 1))).all()
assert (df_lots_m['id'].astype(int) == list(range(1, len(df_lots_m) + 1))).all()
assert df_barangs_m['number'].is_unique
assert df_lots_m['number'].is_unique

print(f"[SUCCESS] All complete ELEK category assertions passed! Master barangs: {len(df_barangs_m):,}, units: {len(df_units_m):,}.")


Verifying Complete ELEK Category Masters & Consistency

[SUCCESS] All complete ELEK category assertions passed! Master barangs: 888, units: 6,941.
